# Hybrid HMM-LSTM model for multi-horizon S&P 500 return forecastingThis notebook implements an end-to-end pipeline for **probabilistic (quantile) forecasting** of S&P 500 returns over 20, 60 and 120 trading-day horizons.The approach combines two components:1. A **Gaussian-mixture Hidden Markov Model (GMM-HMM)** that identifies latent market regimes (bull / neutral / bear) and produces, for each day, the *posterior probability* of being in each regime.2. A **quantile LSTM** that consumes the macro-financial features together with the HMM regime posteriors (used as soft, continuous inputs) and predicts the 5th, 25th, 50th, 75th and 95th percentiles of future cumulative returns.The pipeline includes walk-forward cross-validation, multi-seed ensembling, calibration diagnostics, GARCH baselines, formal statistical tests (Kupiec, Christoffersen, Berkowitz, Diebold-Mariano), a regime-conditional analysis, an ablation study and an economic evaluation.---**How to run.** Set the `FRED_API_KEY` environment variable (free key from https://fred.stlouisfed.org/) and, optionally, `HMM_LSTM_DIR` to choose the output folder. All intermediate artefacts (datasets, HMM states/posteriors, predictions, metrics) are written as CSV files in that folder and reused by later stages.

## Stage 0 — Setup and working directory

In [ ]:
import os# Output / working directory (override with the HMM_LSTM_DIR env var if desired).WORK_DIR = os.environ.get("HMM_LSTM_DIR", "outputs")FIG_DIR = os.path.join(WORK_DIR, "figures")os.makedirs(FIG_DIR, exist_ok=True)os.chdir(WORK_DIR)# FRED API key is read from the environment, never hardcoded.#   export FRED_API_KEY="your_key_here"FRED_API_KEY = os.getenv("FRED_API_KEY", "").strip()print("Working directory:", os.getcwd())

## Stage 0b — Dependencies

In [ ]:
# Install once if running on a fresh environment (e.g. Google Colab):# !pip install -q yfinance fredapi hmmlearn arch statsmodels scikit-learn tensorflow

## Stage 1 — Data acquisition and preprocessingDownloads market and macro series, builds the feature set (returns, realised moments, VIX term-structure and sentiment, commodity and FX returns, yield differentials), standardises it and saves `dataset_standardized.csv`.

In [ ]:
# Requirements: yfinance, pandas, numpy, scikit-learn, scipy, statsmodels, joblib, fredapi (optional)import osimport joblibimport yfinance as yfimport pandas as pdimport numpy as npfrom sklearn.preprocessing import StandardScalerfrom sklearn.decomposition import PCAfrom statsmodels.stats.outliers_influence import variance_inflation_factorfrom scipy.stats import skew, kurtosistry:    from fredapi import Fredexcept ImportError:    Fred = None# CONFIG / PARAMETERSSTART = "2000-01-01"END = "2025-10-01"WINDOWS = [20, 60, 120]TRAIN_PROP = 0.8# Use env var / Colab secret; do NOT hardcode in code.FRED_API_KEY = os.getenv("FRED_API_KEY", "").strip()print("Downloading market data (single multi-ticker call)...")tickers = ["^GSPC", "^VIX", "^VIX3M", "CL=F", "GC=F", "DX-Y.NYB"]raw = yf.download(    tickers,    start=START,    end=END,    group_by="column",    auto_adjust=False,    progress=False)def get_col(df: pd.DataFrame, colname: str, ticker: str) -> pd.Series:    """    yfinance multi-ticker download returns MultiIndex columns.    This helper tries Adj Close first, then Close as fallback.    """    try:        return df[colname][ticker]    except Exception:        if colname == "Adj Close" and "Close" in df.columns:            return df["Close"][ticker]        raiseprint("Building base dataframe...")base = pd.DataFrame(index=raw.index)base["SP500"] = get_col(raw, "Adj Close", "^GSPC").astype(float)base["SP500_Volume"] = raw["Volume"]["^GSPC"].astype(float)base["VIX"] = get_col(raw, "Adj Close", "^VIX").astype(float)base["VIX3M"] = get_col(raw, "Adj Close", "^VIX3M").astype(float)base["Oil"] = get_col(raw, "Adj Close", "CL=F").astype(float)base["Gold"] = get_col(raw, "Adj Close", "GC=F").astype(float)base["DXY"] = get_col(raw, "Adj Close", "DX-Y.NYB").astype(float)# Yields from FRED (optional)if FRED_API_KEY and Fred is not None:    print("Downloading yields from FRED...")    fred = Fred(api_key=FRED_API_KEY)    y10 = fred.get_series("DGS10", start=START, end=END).rename("Y10")    y5 = fred.get_series("DGS5", start=START, end=END).rename("Y5")    yields = pd.concat([y10, y5], axis=1)    yields.index = pd.to_datetime(yields.index)    yields = yields.reindex(base.index).ffill()    base["Y10"] = yields["Y10"].astype(float) / 100.0    base["Y5"] = yields["Y5"].astype(float) / 100.0    base["Yield_Diff"] = base["Y10"] - base["Y5"]else:    if Fred is None:        print("Warning: fredapi not installed. Skipping FRED yields.")    else:        print("Warning: FRED_API_KEY not set. Skipping FRED yields.")    base["Y10"] = np.nan    base["Y5"] = np.nan    base["Yield_Diff"] = np.nanbase = base.ffill().dropna().copy()print(f"Base observations after align/ffill/dropna: {len(base)} rows")# LAG MAPPING (avoid leakage from different close/publish times)lag_one_day = ["VIX", "VIX3M", "Oil", "Gold", "DXY", "Y10", "Y5", "Yield_Diff"]for col in lag_one_day:    base[col + "_lag1"] = base[col].shift(1)# FEATURE ENGINEERING (multi-window)print("Computing features for windows:", WINDOWS)df = base.copy()df["ret_1d"] = np.log(df["SP500"] / df["SP500"].shift(1))for w in WINDOWS:    suffix = f"_{w}d"    r = df["ret_1d"].rolling(w)    df[f"ret{suffix}"] = r.sum()    df[f"rv{suffix}"] = r.std() * np.sqrt(252)    df[f"vol_trend{suffix}"] = df["SP500_Volume"].rolling(w).mean()    df[f"momentum{suffix}"] = df["SP500"] / df["SP500"].shift(w) - 1.0    # Prefer np.nan over artificial 0.0 when stats are undefined    df[f"skew{suffix}"] = r.apply(lambda x: skew(x, bias=False) if np.all(np.isfinite(x)) else np.nan, raw=False)    df[f"kurt{suffix}"] = r.apply(lambda x: kurtosis(x, fisher=True, bias=False) if np.all(np.isfinite(x)) else np.nan, raw=False)# Macro returns (lagged)df["oil_ret"] = np.log(df["Oil_lag1"] / df["Oil_lag1"].shift(1))df["gold_ret"] = np.log(df["Gold_lag1"] / df["Gold_lag1"].shift(1))df["dxy_ret"] = np.log(df["DXY_lag1"] / df["DXY_lag1"].shift(1))# VIX features (lagged)df["VIX_term_lag1"] = (df["VIX_lag1"] / df["VIX3M_lag1"]).replace([np.inf, -np.inf], np.nan)df["VIX_sent_lag1"] = (df["VIX_lag1"] - df["VIX_lag1"].rolling(30).mean()) / df["VIX_lag1"].rolling(30).std()# Yields feature already lagged# df["Yield_Diff_lag1"] exists from basedf = df.dropna().copy()print(f"Observations after feature engineering and dropna: {len(df)} rows")# DEFINE CANDIDATE FEATURE LISTcandidate_features = []for w in WINDOWS:    suf = f"_{w}d"    candidate_features += [f"ret{suf}", f"rv{suf}", f"vol_trend{suf}", f"momentum{suf}", f"skew{suf}", f"kurt{suf}"]candidate_features += ["oil_ret", "gold_ret", "dxy_ret", "Yield_Diff_lag1", "VIX_lag1", "VIX_term_lag1", "VIX_sent_lag1"]candidate_features = [c for c in candidate_features if c in df.columns]print(f"Total candidate features: {len(candidate_features)}")# TRAIN/TEST SPLIT (temporal)split_idx = int(len(df) * TRAIN_PROP)df_train = df.iloc[:split_idx].copy()df_test = df.iloc[split_idx:].copy()print(f"Train rows: {len(df_train)}, Test rows: {len(df_test)}")# STANDARDIZE (fit on train only)scaler = StandardScaler().fit(df_train[candidate_features].values)df_std = df.copy()df_std[candidate_features] = scaler.transform(df[candidate_features].values)joblib.dump(scaler, "scaler_preproc.joblib")df_std.to_csv("dataset_standardized.csv", index=True)print("Saved scaler -> scaler_preproc.joblib and dataset -> dataset_standardized.csv")# PCA (diagnostic) on TRAIN onlypca = PCA(n_components=min(len(candidate_features), 10))pca.fit(df_train[candidate_features].values)explained = pca.explained_variance_ratio_cum_explained = np.cumsum(explained)print("PCA explained variance ratio:")for i, (e, ce) in enumerate(zip(explained, cum_explained), 1):    print(f" PC{i}: {e:.3f}, cumulative: {ce:.3f}")loadings = pd.DataFrame(    pca.components_.T,    index=candidate_features,    columns=[f"PC{i+1}" for i in range(pca.n_components_)])loadings.to_csv("pca_loadings.csv")# VIF (diagnostic) on TRAIN# Note: statsmodels VIF assumes design matrix includes a constant in classic regression setups,# but for pure multicollinearity screening it's still commonly used. Consider adding constant if desired.X = df_train[candidate_features].copy()X = X.replace([np.inf, -np.inf], np.nan).dropna()vif_df = pd.DataFrame({    "feature": X.columns,    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]}).sort_values("VIF", ascending=False)vif_df.to_csv("vif_train.csv", index=False)# Correlations (diagnostic) on TRAINcorr = df_train[candidate_features].corr().abs()corr.to_csv("corr_train.csv")print("Saved diagnostics: pca_loadings.csv, vif_train.csv, corr_train.csv")

## Stage 2 — HMM feature preparation per horizonBuilds the per-window feature matrices used to fit the HMM and runs a final VIF check on each window.

In [ ]:
# Prepara dataset per HMM su tutte e tre le finestre temporali# Requirements: pandas, numpy, sklearn, statsmodels, joblibimport pandas as pdimport numpy as npfrom sklearn.preprocessing import StandardScalerfrom statsmodels.stats.outliers_influence import variance_inflation_factorimport joblib# CONFIGURAZIONE FEATURES FINALI# Rimuoviamo momentum (ridondante), sostituiamo rv con VIX_lag1FINAL_FEATURES = {    '20d': ['ret_20d', 'VIX_lag1', 'vol_trend_20d', 'skew_20d', 'kurt_20d',            'oil_ret', 'gold_ret', 'dxy_ret', 'Yield_Diff_lag1',            'VIX_term_lag1', 'VIX_sent_lag1'],    '60d': ['ret_60d', 'VIX_lag1', 'vol_trend_60d', 'skew_60d', 'kurt_60d',            'oil_ret', 'gold_ret', 'dxy_ret', 'Yield_Diff_lag1',            'VIX_term_lag1', 'VIX_sent_lag1'],    '120d': ['ret_120d', 'VIX_lag1', 'vol_trend_120d', 'skew_120d', 'kurt_120d',             'oil_ret', 'gold_ret', 'dxy_ret', 'Yield_Diff_lag1',             'VIX_term_lag1', 'VIX_sent_lag1']}TRAIN_PROP = 0.8# CARICAMENTO DATASET STANDARDIZZATOprint("Caricando dataset standardizzato...")df_std = pd.read_csv("dataset_standardized.csv", index_col=0, parse_dates=True)print(f"Dataset caricato: {len(df_std)} righe\n")# PROCESSING PER OGNI FINESTRAresults = {}for window, features in FINAL_FEATURES.items():    print("="*80)    print(f"PROCESSING FINESTRA: {window}")    print("="*80)    # Verifica che tutte le features esistano    missing = [f for f in features if f not in df_std.columns]    if missing:        print(f"⚠️  Features mancanti: {missing}")        continue    # Estrai subset    df_window = df_std[features].copy().dropna()    print(f"Features selezionate: {len(features)}")    print(f"Osservazioni disponibili: {len(df_window)}\n")    # Train/test split temporale    split_idx = int(len(df_window) * TRAIN_PROP)    X_train = df_window.iloc[:split_idx].copy()    X_test = df_window.iloc[split_idx:].copy()    print(f"Train samples: {len(X_train)}")    print(f"Test samples: {len(X_test)}\n")    # VERIFICA VIF SUL DATASET PULITO    print("-"*80)    print("VIF VERIFICATION (dataset pulito)")    print("-"*80)    vif_data = pd.DataFrame({        "feature": features,        "VIF": [variance_inflation_factor(X_train.values, i)                for i in range(len(features))]    }).sort_values("VIF", ascending=False)    print(vif_data.to_string(index=False))    # Conta features con VIF problematico    high_vif = vif_data[vif_data["VIF"] > 10]    if len(high_vif) > 0:        print(f"\n⚠️  Features con VIF > 10: {len(high_vif)}")        print(high_vif.to_string(index=False))    else:        print("\n✓ Nessuna feature con VIF > 10")    medium_vif = vif_data[(vif_data["VIF"] > 5) & (vif_data["VIF"] <= 10)]    if len(medium_vif) > 0:        print(f"\n⚠️  Features con VIF 5-10: {len(medium_vif)}")        print(medium_vif.to_string(index=False))    # STATISTICHE DESCRITTIVE    print("\n" + "-"*80)    print("STATISTICHE DESCRITTIVE (Train set)")    print("-"*80)    desc = X_train.describe().T[['mean', 'std', 'min', 'max']]    print(desc.to_string())    # SALVATAGGIO    # Salva dataset per HMM    X_train.to_csv(f"X_train_{window}_hmm.csv")    X_test.to_csv(f"X_test_{window}_hmm.csv")    # Salva anche versione con indice originale per allineamento temporale    df_window.to_csv(f"X_full_{window}_hmm.csv")    # Salva VIF    vif_data.to_csv(f"vif_final_{window}.csv", index=False)    print(f"\n✓ File salvati:")    print(f"  - X_train_{window}_hmm.csv")    print(f"  - X_test_{window}_hmm.csv")    print(f"  - X_full_{window}_hmm.csv")    print(f"  - vif_final_{window}.csv")    # Salva nei risultati    results[window] = {        'n_features': len(features),        'n_train': len(X_train),        'n_test': len(X_test),        'max_vif': vif_data['VIF'].max(),        'mean_vif': vif_data['VIF'].mean(),        'n_vif_high': len(high_vif)    }    print("\n")# SUMMARY FINALEprint("="*80)print("SUMMARY COMPARATIVO")print("="*80)summary = pd.DataFrame(results).Tsummary.columns = ['N Features', 'N Train', 'N Test', 'Max VIF', 'Mean VIF', 'N VIF>10']print(summary.to_string())summary.to_csv("hmm_preparation_summary.csv")print("\n✓ Summary salvato in: hmm_preparation_summary.csv")# NEXT STEPSprint("\n" + "="*80)print("NEXT STEPS: HMM FITTING")print("="*80)print("""Per ogni finestra, hai ora dataset pronti per HMM:1. SCELTA NUMERO STATI (n_components):   - Inizia con 2-3 stati (bull/bear o bull/neutral/bear)   - Valuta con BIC/AIC quale numero è ottimale2. FIT HMM:   from hmmlearn import hmm   # Esempio per finestra 60d   model = hmm.GaussianHMM(n_components=3, covariance_type="full",                           n_iter=1000, random_state=42)   model.fit(X_train_60d)   # Predici stati   states_train = model.predict(X_train_60d)   states_test = model.predict(X_test_60d)3. VALUTAZIONE:   - Visualizza time series degli stati   - Calcola statistiche per regime (mean return, volatility per stato)   - Confronta le tre finestre: quale separa meglio i regimi?4. INTEGRAZIONE CON LSTM:   - Usa gli stati predetti come feature aggiuntiva per LSTM   - Oppure: allena LSTM separati per ogni regimeFile pronti:  - X_train_{20d,60d,120d}_hmm.csv  - X_test_{20d,60d,120d}_hmm.csv  - X_full_{20d,60d,120d}_hmm.csv""")print("="*80)print("RACCOMANDAZIONI FINALI")print("="*80)print("""✓ Features pulite: rimossi momentum e rv, mantenuto VIX_lag1✓ VIF verificati: dovresti vedere valori molto più bassi✓ Dataset pronti per HMM su tutte e tre le finestreSuggerimenti:- Confronta HMM su 20d/60d/120d: quale identifica meglio bull/bear/neutral?- La finestra 60d è spesso il miglior compromesso- Puoi anche fare ensemble: usa stati da tutte e tre le finestre come input LSTM""")

## Stage 3 — Diagnostic tests (stationarity, normality, outliers)Augmented Dickey-Fuller, Jarque-Bera and z-score outlier checks on each window, motivating the minimal stationarity corrections applied next.

In [ ]:
# Test di stazionarietà, normalità e outlier per tutte le finestreimport pandas as pdimport numpy as npfrom statsmodels.tsa.stattools import adfullerfrom scipy.stats import jarque_bera, zscore# CONFIGURAZIONEWINDOWS = ['20d', '60d', '120d']# FUNZIONE PER TEST DIAGNOSTICIdef run_diagnostics(X_train, window_name):    """Esegue tutti i test diagnostici su un dataset"""    print("="*80)    print(f"DIAGNOSTICS FOR WINDOW: {window_name}")    print("="*80)    # 1. Test di stazionarietà (ADF)    print("\n" + "-"*80)    print("1. STATIONARITY TEST (Augmented Dickey-Fuller)")    print("-"*80)    print("H0: series has unit root (non-stationary)")    print("p-value < 0.05 → reject H0 → stationary\n")    adf_results = []    for col in X_train.columns:        result = adfuller(X_train[col].dropna())        adf_results.append({            'feature': col,            'ADF_stat': result[0],            'p_value': result[1],            'stationary': 'YES' if result[1] < 0.05 else 'NO'        })        print(f"{col:20s}: ADF={result[0]:7.3f}, p-value={result[1]:.4f} → {adf_results[-1]['stationary']}")    adf_df = pd.DataFrame(adf_results)    non_stationary = adf_df[adf_df['stationary'] == 'NO']    if len(non_stationary) > 0:        print(f"\n⚠️  NON-STATIONARY FEATURES ({len(non_stationary)}):")        for _, row in non_stationary.iterrows():            print(f"  - {row['feature']} (p-value={row['p_value']:.4f})")    else:        print("\n✓ All features are stationary")    # 2. Test di normalità (Jarque-Bera)    print("\n" + "-"*80)    print("2. NORMALITY TEST (Jarque-Bera)")    print("-"*80)    print("H0: data is normally distributed")    print("p-value < 0.05 → reject H0 → NOT normal\n")    jb_results = []    for col in X_train.columns:        stat, pval = jarque_bera(X_train[col])        skewness = X_train[col].skew()        kurtosis = X_train[col].kurt()        jb_results.append({            'feature': col,            'JB_stat': stat,            'p_value': pval,            'skew': skewness,            'kurt': kurtosis,            'normal': 'YES' if pval >= 0.05 else 'NO'        })        print(f"{col:20s}: JB p-val={pval:.4f}, skew={skewness:6.2f}, kurt={kurtosis:6.2f} → {jb_results[-1]['normal']}")    jb_df = pd.DataFrame(jb_results)    # Identifica features con problemi severi    extreme_skew = jb_df[np.abs(jb_df['skew']) > 2]    extreme_kurt = jb_df[jb_df['kurt'] > 5]    if len(extreme_skew) > 0:        print(f"\n⚠️  EXTREME SKEWNESS (|skew| > 2): {len(extreme_skew)}")        for _, row in extreme_skew.iterrows():            print(f"  - {row['feature']}: skew={row['skew']:.2f}")    if len(extreme_kurt) > 0:        print(f"\n⚠️  HEAVY TAILS (kurt > 5): {len(extreme_kurt)}")        for _, row in extreme_kurt.iterrows():            print(f"  - {row['feature']}: kurt={row['kurt']:.2f}")    # 3. Outlier detection    print("\n" + "-"*80)    print("3. OUTLIER DETECTION (|z-score| > 3)")    print("-"*80)    z_scores = np.abs(zscore(X_train))    outliers = (z_scores > 3).sum(axis=0)    outlier_pct = (outliers / len(X_train)) * 100    outlier_data = pd.DataFrame({        'feature': X_train.columns,        'n_outliers': outliers,        'pct_outliers': outlier_pct    }).sort_values('n_outliers', ascending=False)    print(outlier_data.to_string(index=False))    severe_outliers = outlier_data[outlier_data['pct_outliers'] > 3]    if len(severe_outliers) > 0:        print(f"\n⚠️  HIGH OUTLIER RATE (>3%): {len(severe_outliers)}")        for _, row in severe_outliers.iterrows():            print(f"  - {row['feature']}: {row['n_outliers']} outliers ({row['pct_outliers']:.1f}%)")    else:        print("\n✓ Outlier rate acceptable (<3% for all features)")    # 4. Statistiche descrittive    print("\n" + "-"*80)    print("4. DESCRIPTIVE STATISTICS")    print("-"*80)    desc = X_train.describe().T[['mean', 'std', 'min', 'max']]    print(desc.to_string())    # Verifica standardizzazione    mean_ok = (np.abs(desc['mean']) < 0.01).all()    std_ok = ((desc['std'] > 0.95) & (desc['std'] < 1.05)).all()    if mean_ok and std_ok:        print("\n✓ Standardization OK (mean≈0, std≈1)")    else:        print("\n⚠️  Standardization issues detected")    # Salva risultati    adf_df.to_csv(f'diagnostics_adf_{window_name}.csv', index=False)    jb_df.to_csv(f'diagnostics_normality_{window_name}.csv', index=False)    outlier_data.to_csv(f'diagnostics_outliers_{window_name}.csv', index=False)    print(f"\n✓ Results saved:")    print(f"  - diagnostics_adf_{window_name}.csv")    print(f"  - diagnostics_normality_{window_name}.csv")    print(f"  - diagnostics_outliers_{window_name}.csv")    return {        'adf': adf_df,        'normality': jb_df,        'outliers': outlier_data    }# ESEGUI TEST PER TUTTE LE FINESTREall_results = {}for window in WINDOWS:    # Carica dataset    try:        X_train = pd.read_csv(f'X_train_{window}_hmm.csv', index_col=0, parse_dates=True)        print(f"\n✓ Loaded X_train_{window}_hmm.csv ({len(X_train)} rows)")        # Esegui diagnostici        results = run_diagnostics(X_train, window)        all_results[window] = results    except FileNotFoundError:        print(f"\n⚠️  File X_train_{window}_hmm.csv not found. Skipping.")        continue    print("\n")# SUMMARY COMPARATIVOprint("\n" + "="*80)print("COMPARATIVE SUMMARY")print("="*80)summary_data = []for window in WINDOWS:    if window in all_results:        adf_df = all_results[window]['adf']        jb_df = all_results[window]['normality']        outlier_df = all_results[window]['outliers']        summary_data.append({            'Window': window,            'N Features': len(adf_df),            'Non-stationary': (adf_df['stationary'] == 'NO').sum(),            'Non-normal': (jb_df['normal'] == 'NO').sum(),            'Extreme skew (|s|>2)': (np.abs(jb_df['skew']) > 2).sum(),            'Heavy tails (k>5)': (jb_df['kurt'] > 5).sum(),            'Total outliers': outlier_df['n_outliers'].sum(),            'Max outlier %': outlier_df['pct_outliers'].max()        })summary_df = pd.DataFrame(summary_data)print("\n" + summary_df.to_string(index=False))summary_df.to_csv('diagnostics_summary_all_windows.csv', index=False)print("\n✓ Summary saved: diagnostics_summary_all_windows.csv")# RACCOMANDAZIONIprint("\n" + "="*80)print("RECOMMENDATIONS")print("="*80)for window in WINDOWS:    if window not in all_results:        continue    print(f"\n{window.upper()}:")    adf_df = all_results[window]['adf']    jb_df = all_results[window]['normality']    outlier_df = all_results[window]['outliers']    # Check non-stazionarietà    non_stat = adf_df[adf_df['stationary'] == 'NO']    if len(non_stat) > 0:        print(f"  ⚠️  {len(non_stat)} non-stationary feature(s):")        for feat in non_stat['feature'].values:            print(f"     → Consider differencing or removing: {feat}")    # Check code pesanti    heavy_tails = jb_df[jb_df['kurt'] > 8]    if len(heavy_tails) > 0:        print(f"  ⚠️  {len(heavy_tails)} feature(s) with very heavy tails (kurt>8):")        for feat in heavy_tails['feature'].values:            print(f"     → Use GMM-HMM or winsorize: {feat}")    # Check outlier elevati    high_outliers = outlier_df[outlier_df['pct_outliers'] > 3]    if len(high_outliers) > 0:        print(f"  ⚠️  {len(high_outliers)} feature(s) with high outlier rate (>3%):")        for _, row in high_outliers.iterrows():            print(f"     → Winsorize: {row['feature']} ({row['pct_outliers']:.1f}%)")    if len(non_stat) == 0 and len(heavy_tails) == 0 and len(high_outliers) == 0:        print("  ✓ No major issues detected")print("\n" + "="*80)

## Stage 4 — GMM-HMM fitting (regimes and posteriors)Applies minimal stationarity fixes, fits a 3-state Gaussian-mixture HMM per horizon and saves the regime sequences, the **posterior probabilities** (`hmm_proba_*`, the soft inputs used later by the LSTM) and the state characterisations.

In [ ]:
# Correzioni minime + fit GMM-HMM su tutte le finestre# Requirements: hmmlearn, pandas, numpy, matplotlib, seabornimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom hmmlearn import hmmimport warningswarnings.filterwarnings('ignore')# CONFIGURAZIONEWINDOWS = ['20d', '60d', '120d']N_STATES = 3  # Testeremo diversi numeri di statiN_MIX = 2  # Mixture components per gestire code pesantiTRAIN_PROP = 0.8RANDOM_STATE = 42# STEP 1: CORREZIONI MINIME (SOLO STAZIONARIETÀ)print("="*80)print("STEP 1: APPLICAZIONE CORREZIONI MINIME")print("="*80)# Carica dataset originaledf_full = pd.read_csv("dataset_standardized.csv", index_col=0, parse_dates=True)print(f"✓ Dataset caricato: {len(df_full)} righe")# Differenzia Yield_Diff_lag1 (non stazionaria su tutte le finestre)print("\n1. Differenziando Yield_Diff_lag1...")df_full['Yield_Diff_change'] = df_full['Yield_Diff_lag1'].diff()# Differenzia vol_trend_120d (borderline non stazionaria)print("2. Differenziando vol_trend_120d...")df_full['vol_trend_120d_change'] = df_full['vol_trend_120d'].diff()print("✓ Correzioni applicate (mantenuti tutti gli outlier e code pesanti)\n")

In [ ]:
# STEP 2: PREPARA DATASET PER OGNI FINESTRAdatasets = {}for window in WINDOWS:    print(f"\nPreparando dataset per finestra {window}...")    # Definisci features con correzioni    if window == '20d':        features = ['ret_20d', 'VIX_lag1', 'vol_trend_20d', 'skew_20d', 'kurt_20d',                   'oil_ret', 'gold_ret', 'dxy_ret', 'Yield_Diff_change',                   'VIX_term_lag1', 'VIX_sent_lag1']    elif window == '60d':        features = ['ret_60d', 'VIX_lag1', 'vol_trend_60d', 'skew_60d', 'kurt_60d',                   'oil_ret', 'gold_ret', 'dxy_ret', 'Yield_Diff_change',                   'VIX_term_lag1', 'VIX_sent_lag1']    else:  # 120d        features = ['ret_120d', 'VIX_lag1', 'vol_trend_120d_change', 'skew_120d', 'kurt_120d',                   'oil_ret', 'gold_ret', 'dxy_ret', 'Yield_Diff_change',                   'VIX_term_lag1', 'VIX_sent_lag1']    # Estrai subset    df_window = df_full[features].copy().dropna()    # Train/test split temporale    split_idx = int(len(df_window) * TRAIN_PROP)    X_train = df_window.iloc[:split_idx].values    X_test = df_window.iloc[split_idx:].values    # Salva indici per visualizzazioni    train_dates = df_window.index[:split_idx]    test_dates = df_window.index[split_idx:]    datasets[window] = {        'X_train': X_train,        'X_test': X_test,        'train_dates': train_dates,        'test_dates': test_dates,        'features': features,        'df_full': df_window    }    print(f"  ✓ Train: {len(X_train)}, Test: {len(X_test)}")# STEP 3: FIT GMM-HMM (3 stati fissi) + salvataggi per LSTMprint("\n" + "="*80)print("STEP 3: FITTING GMM-HMM (fixed 3 states)")print("="*80)N_STATES = 3results = {}for window in WINDOWS:    print(f"\n{'='*80}")    print(f"WINDOW: {window} | n_states={N_STATES}, n_mix={N_MIX}")    print(f"{'='*80}")    X_train = datasets[window]['X_train']    X_test = datasets[window]['X_test']    train_dates = datasets[window]['train_dates']    test_dates = datasets[window]['test_dates']    df_window = datasets[window]['df_full']    features = datasets[window]['features']    best_score = -np.inf    best_model = None    for seed in range(5):        try:            model = hmm.GMMHMM(                n_components=N_STATES,                n_mix=N_MIX,                covariance_type="full",                n_iter=1000,                random_state=RANDOM_STATE + seed,                verbose=False            )            model.fit(X_train)            score = model.score(X_train)            if score > best_score:                best_score = score                best_model = model        except Exception as e:            print(f"  ⚠️ Seed {seed} failed: {e}")            continue    if best_model is None:        print("  ❌ All fits failed.")        continue    # Stati (Viterbi) su train e test separati    states_train = best_model.predict(X_train)    states_test = best_model.predict(X_test)    # Probabilità posteriori per stato (utile per LSTM come feature continua)    proba_train = best_model.predict_proba(X_train)    proba_test = best_model.predict_proba(X_test)    train_ll = best_model.score(X_train)    test_ll = best_model.score(X_test)    print(f"\n  Train log-likelihood: {train_ll:.2f}")    print(f"  Test  log-likelihood: {test_ll:.2f}")    # Salva risultati base    results[window] = {        "model": best_model,        "states_train": states_train,        "states_test": states_test,        "proba_train": proba_train,        "proba_test": proba_test,        "train_ll": train_ll,        "test_ll": test_ll,        "features": features    }    # ---- Salvataggi su CSV indicizzati per merge con dataset LSTM ----    # Stati discreti    pd.Series(states_train, index=train_dates, name="state").to_csv(f"hmm_states_train_{window}_3states.csv")    pd.Series(states_test, index=test_dates, name="state").to_csv(f"hmm_states_test_{window}_3states.csv")    # Probabilità (3 colonne)    pd.DataFrame(        proba_train, index=train_dates,        columns=[f"p_state_{i}" for i in range(N_STATES)]    ).to_csv(f"hmm_proba_train_{window}_3states.csv")    pd.DataFrame(        proba_test, index=test_dates,        columns=[f"p_state_{i}" for i in range(N_STATES)]    ).to_csv(f"hmm_proba_test_{window}_3states.csv")    # Characterization (sul train)    ret_col = [c for c in df_window.columns if "ret_" in c][0]    ret_idx = [i for i, f in enumerate(features) if "ret_" in f][0]    vix_idx = [i for i, f in enumerate(features) if f == "VIX_lag1"][0]    state_stats = []    for s in range(N_STATES):        mask = states_train == s        state_stats.append({            "State": s,            "Count": int(mask.sum()),            "Pct": float(mask.mean() * 100),            "Mean_Return": float(X_train[mask, ret_idx].mean()),            "Std_Return": float(X_train[mask, ret_idx].std()),            "Mean_VIX": float(X_train[mask, vix_idx].mean()),            "Std_VIX": float(X_train[mask, vix_idx].std()),        })    pd.DataFrame(state_stats).to_csv(f"state_characteristics_{window}_3states.csv", index=False)print("\n" + "="*80)print("DONE: fitted fixed 3-state GMM-HMM for all windows (where fit succeeded).")print("="*80)

## Stage 5 — Regime visualisation

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom matplotlib.lines import Line2DWINDOWS = ["20d", "60d", "120d"]N_STATES = 3STATE_LABELS = {    "20d":  {0: "Bull (calm)", 2: "Neutral", 1: "Bear (crisis)"},    "60d":  {1: "Bull (calm)", 0: "Neutral", 2: "Bear (crisis)"},    "120d": {2: "Bull (calm)", 1: "Neutral", 0: "Bear (crisis)"},}# Colori fissi PER REGIME (coerenti su tutte le finestre)REGIME_COLORS = {    "Bear (crisis)": "#440154",  # viola scuro (stile viridis low)    "Neutral": "#21918c",        # verde/teal (viridis mid)    "Bull (calm)": "#fde725",    # giallo (viridis high)}# Ordine fisso in legenda (uguale per tutte le finestre)REGIME_ORDER = ["Bear (crisis)", "Neutral", "Bull (calm)"]# Dataset con returns per plottare "returns vs regime"df_std = pd.read_csv("dataset_standardized.csv", index_col=0, parse_dates=True)for window in WINDOWS:    labels_map = STATE_LABELS[window]  # state_id -> regime label    state_ids = list(range(N_STATES))  # [0,1,2]    # --- carica stati train/test e concatena    s_tr = pd.read_csv(f"hmm_states_train_{window}_3states.csv", index_col=0, parse_dates=True).iloc[:, 0]    s_te = pd.read_csv(f"hmm_states_test_{window}_3states.csv", index_col=0, parse_dates=True).iloc[:, 0]    states = pd.concat([s_tr, s_te]).sort_index()    split_date = s_tr.index.max()    # (A) Timeline stati -> colorata per REGIME, non per state_id    # Converti la sequenza numerica in label economiche    regimes = states.map(labels_map)    fig, ax = plt.subplots(figsize=(14, 4))    # scatter per ciascun regime (così i colori restano coerenti)    for reg in REGIME_ORDER:        m = regimes.values == reg        ax.scatter(            states.index[m],            states.values[m],          # mantengo y=state_id (0/1/2) per vedere i livelli discreti            s=6,            alpha=0.7,            color=REGIME_COLORS[reg],            label=reg        )    ax.axvline(split_date, color="red", linestyle="--", linewidth=2, label="Train/Test split")    ax.set_title(f"{window} – Market regimes (3 states, decoded separately)")    ax.set_ylabel("Market regime")    # tick label sull'asse y: da state_id -> label (può cambiare tra finestre)    ax.set_yticks(state_ids)    ax.set_yticklabels([labels_map[i] for i in state_ids])    ax.grid(True, alpha=0.3)    # legenda: regimi in ordine fisso + split    handles = [Line2D([0], [0], marker="o", linestyle="None", markersize=6,                      markerfacecolor=REGIME_COLORS[r], markeredgecolor="none", label=r)               for r in REGIME_ORDER]    handles.append(Line2D([0], [0], color="red", linestyle="--", linewidth=2, label="Train/Test split"))    ax.legend(handles=handles, loc="lower left", frameon=True)    plt.tight_layout()    plt.savefig(f"plot_states_timeline_{window}.png", dpi=150, bbox_inches="tight")    plt.close()    # (B) Returns colorati per REGIME (coerenti tra finestre)    ret_col = f"ret_{window}"    if ret_col not in df_std.columns:        print(f"[{window}] Missing {ret_col} in dataset_standardized.csv, skipping return plot.")        continue    aligned = pd.DataFrame({        "ret": df_std.loc[states.index, ret_col],        "regime": regimes    }).dropna()    fig, ax = plt.subplots(figsize=(14, 5))    for reg in REGIME_ORDER:        m = aligned["regime"].values == reg        ax.scatter(            aligned.index[m],            aligned["ret"].values[m],            s=8,            alpha=0.6,            color=REGIME_COLORS[reg],            label=reg        )    ax.axvline(split_date, color="red", linestyle="--", linewidth=2, label="Train/Test split")    ax.set_title(f"{window} – Returns colored by market regime")    ax.set_ylabel(ret_col)    ax.grid(True, alpha=0.3)    ax.legend(loc="lower left", frameon=True)    plt.tight_layout()    plt.savefig(f"plot_returns_by_regime_{window}.png", dpi=150, bbox_inches="tight")    plt.close()print("Saved plots: plot_states_timeline_*.png and plot_returns_by_regime_*.png")

## Stage 6 — HMM diagnosticsMulti-seed stability, Ljung-Box residual autocorrelation, state durations and overlap with known historical crisis windows.

In [ ]:
# Diagnostiche avanzate per GMM-HMM (3 stati) su 20d/60d/120d# Compatibile con: dataset_standardized.csv + trasformazioni minime (diff)# Requirements: hmmlearn, pandas, numpy, matplotlib, seaborn, statsmodelsimport warningswarnings.filterwarnings("ignore")import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom hmmlearn import hmmfrom statsmodels.stats.diagnostic import acorr_ljungbox# CONFIGURATIONWINDOWS = ["20d", "60d", "120d"]N_STATES = 3N_MIX = 2TRAIN_PROP = 0.8RANDOM_STATE = 42N_SEEDS = 5LJUNG_LAGS = 20AMBIGUOUS_THRESHOLD = 0.6# Event windows: usiamo solo l'intervallo, senza richiedere che start/end siano trading daysKNOWN_EVENTS = [    ("2007-10-09", "2009-03-09", "2008 Financial Crisis"),    ("2020-02-19", "2020-03-23", "COVID-19 Crash"),    ("2022-01-03", "2022-10-13", "2022 Bear Market"),    ("2011-07-22", "2011-10-04", "2011 Debt Ceiling Crisis"),    ("2015-08-10", "2015-09-30", "2015 China Slowdown"),]print("=" * 80)print("HMM DIAGNOSTICS - ALL WINDOWS (GMM-HMM | 3 STATES)")print("=" * 80)# 1. LOAD DATA + TRANSFORMATIONS (coerenti col fitting)print("\n1) Loading dataset...")df_full = pd.read_csv("dataset_standardized.csv", index_col=0, parse_dates=True)# Evita fillna(0): lasciamo NaN e poi dropna nei subsetdf_full["Yield_Diff_change"] = df_full["Yield_Diff_lag1"].diff()df_full["vol_trend_120d_change"] = df_full["vol_trend_120d"].diff()print(f"✓ Rows: {len(df_full)} | from {df_full.index.min().date()} to {df_full.index.max().date()}")def get_features_for_window(window: str):    if window == "20d":        feats = [            "ret_20d", "VIX_lag1", "vol_trend_20d", "skew_20d", "kurt_20d",            "oil_ret", "gold_ret", "dxy_ret", "Yield_Diff_change",            "VIX_term_lag1", "VIX_sent_lag1"        ]        target_feat = "ret_20d"    elif window == "60d":        feats = [            "ret_60d", "VIX_lag1", "vol_trend_60d", "skew_60d", "kurt_60d",            "oil_ret", "gold_ret", "dxy_ret", "Yield_Diff_change",            "VIX_term_lag1", "VIX_sent_lag1"        ]        target_feat = "ret_60d"    else:  # 120d        feats = [            "ret_120d", "VIX_lag1", "vol_trend_120d_change", "skew_120d", "kurt_120d",            "oil_ret", "gold_ret", "dxy_ret", "Yield_Diff_change",            "VIX_term_lag1", "VIX_sent_lag1"        ]        target_feat = "ret_120d"    return feats, target_featdef state_durations(states: np.ndarray, n_states: int):    durations = {i: [] for i in range(n_states)}    cur = int(states[0])    dur = 1    for t in range(1, len(states)):        s = int(states[t])        if s == cur:            dur += 1        else:            durations[cur].append(dur)            cur = s            dur = 1    durations[cur].append(dur)    return durations# Container risultatidiag = {}# 2. FIT (TRAIN ONLY) + DECODE TRAIN/TEST SEPARATELYprint("\n2) Fitting models on TRAIN only (per window) ...")for window in WINDOWS:    feats, target_feat = get_features_for_window(window)    missing = [c for c in feats if c not in df_full.columns]    if missing:        print(f"[{window}] Missing columns: {missing} -> skipping")        continue    df_window = df_full[feats].dropna().copy()    n = len(df_window)    split_idx = int(n * TRAIN_PROP)    df_train = df_window.iloc[:split_idx].copy()    df_test = df_window.iloc[split_idx:].copy()    X_train = df_train.values    X_test = df_test.values    best_ll = -np.inf    best_model = None    best_seed = None    for k in range(N_SEEDS):        seed = RANDOM_STATE + k        try:            model = hmm.GMMHMM(                n_components=N_STATES,                n_mix=N_MIX,                covariance_type="full",                n_iter=1000,                random_state=seed,                verbose=False            )            model.fit(X_train)            ll = model.score(X_train)            if ll > best_ll:                best_ll = ll                best_model = model                best_seed = seed        except Exception as e:            print(f"[{window}] seed {seed} failed: {e}")    if best_model is None:        print(f"[{window}] ❌ all fits failed")        continue    # Decode train/test separati (NO predict su tutta la serie)    states_train = best_model.predict(X_train)    states_test = best_model.predict(X_test)    proba_train = best_model.predict_proba(X_train)  # posterior per stato [web:91]    proba_test = best_model.predict_proba(X_test)    # posterior per stato [web:91]    ll_train = best_model.score(X_train)    ll_test = best_model.score(X_test)    print(f"\n[{window}] best_seed={best_seed} | converged={best_model.monitor_.converged} | iters={best_model.monitor_.iter}")  # monitor_ [web:61]    print(f"[{window}] Train LL={ll_train:.2f} | Test LL={ll_test:.2f}")    diag[window] = {        "model": best_model,        "features": feats,        "target_feat": target_feat,        "df_train": df_train,        "df_test": df_test,        "states_train": states_train,        "states_test": states_test,        "proba_train": proba_train,        "proba_test": proba_test,    }# 3. TRANSITION MATRICES (TRAIN-FIT)print("\n" + "=" * 80)print("3) TRANSITION MATRIX (from train-fit model)")print("=" * 80)fig, axes = plt.subplots(1, len(diag), figsize=(5 * len(diag), 4))if len(diag) == 1:    axes = [axes]for ax, window in zip(axes, diag.keys()):    model = diag[window]["model"]    trans = model.transmat_    trans_df = pd.DataFrame(        trans,        index=[f"S{i}" for i in range(N_STATES)],        columns=[f"S{i}" for i in range(N_STATES)]    )    print(f"\n[{window}] transition matrix:\n{trans_df.to_string()}")    for i in range(N_STATES):        p_ii = float(trans[i, i])        avg_dur = 1.0 / (1.0 - p_ii) if p_ii < 0.999999 else np.inf        print(f"[{window}] State {i}: p_ii={p_ii:.3f} | avg duration ≈ {avg_dur:.1f} days")    sns.heatmap(trans, annot=True, fmt=".3f", cmap="YlOrRd", vmin=0, vmax=1, ax=ax, cbar=False)    ax.set_title(f"{window} transmat")    ax.set_xlabel("To")    ax.set_ylabel("From")plt.tight_layout()plt.savefig("hmm_3states_transition_matrices.png", dpi=150, bbox_inches="tight")plt.close()print("\n✓ Saved: hmm_3states_transition_matrices.png")# 4. STATE DURATIONS (TRAIN and TEST)print("\n" + "=" * 80)print("4) STATE DURATION ANALYSIS (train vs test)")print("=" * 80)for window in diag.keys():    st_tr = diag[window]["states_train"]    st_te = diag[window]["states_test"]    d_tr = state_durations(st_tr, N_STATES)    d_te = state_durations(st_te, N_STATES)    rows = []    for s in range(N_STATES):        for part, d in [("train", d_tr[s]), ("test", d_te[s])]:            if len(d) == 0:                rows.append({"window": window, "split": part, "state": s, "episodes": 0,                             "mean": np.nan, "median": np.nan, "min": np.nan, "max": np.nan})            else:                rows.append({"window": window, "split": part, "state": s, "episodes": len(d),                             "mean": float(np.mean(d)), "median": float(np.median(d)),                             "min": int(np.min(d)), "max": int(np.max(d))})    dur_df = pd.DataFrame(rows)    print(f"\n[{window}] duration stats:\n{dur_df.to_string(index=False)}")    dur_df.to_csv(f"hmm_state_durations_{window}.csv", index=False)print("\n✓ Saved: hmm_state_durations_{window}.csv (x windows)")# 5. RESIDUAL AUTOCORRELATION (Ljung-Box) on TRAINprint("\n" + "=" * 80)print("5) RESIDUAL AUTOCORRELATION (Ljung-Box) on TRAIN residuals")print("=" * 80)for window in diag.keys():    feats = diag[window]["features"]    target_feat = diag[window]["target_feat"]    vix_feat = "VIX_lag1"    X_train = diag[window]["df_train"].values    states_train = diag[window]["states_train"]    idx_target = feats.index(target_feat)    idx_vix = feats.index(vix_feat)    def residuals_by_state(x, states, feat_idx):        res = np.zeros(len(x))        for s in range(N_STATES):            m = states == s            mu = x[m, feat_idx].mean() if m.any() else 0.0            res[m] = x[m, feat_idx] - mu        return res    res_ret = residuals_by_state(X_train, states_train, idx_target)    lb_ret = acorr_ljungbox(res_ret, lags=LJUNG_LAGS, return_df=True)  # columns lb_stat, lb_pvalue [web:126]    sig_ret = int((lb_ret["lb_pvalue"] < 0.05).sum())    res_vix = residuals_by_state(X_train, states_train, idx_vix)    lb_vix = acorr_ljungbox(res_vix, lags=LJUNG_LAGS, return_df=True)  # [web:126]    sig_vix = int((lb_vix["lb_pvalue"] < 0.05).sum())    print(f"\n[{window}] Ljung-Box (TRAIN residuals):")    print(f"  {target_feat}: significant lags p<0.05 = {sig_ret}/{LJUNG_LAGS}")    print(f"  {vix_feat}:    significant lags p<0.05 = {sig_vix}/{LJUNG_LAGS}")    lb_ret.to_csv(f"hmm_ljungbox_{window}_{target_feat}.csv", index=False)    lb_vix.to_csv(f"hmm_ljungbox_{window}_{vix_feat}.csv", index=False)print("\n✓ Saved: hmm_ljungbox_* CSVs")# 6. POSTERIOR CERTAINTY (TRAIN and TEST)print("\n" + "=" * 80)print("6) POSTERIOR PROBABILITY CERTAINTY (train vs test)")print("=" * 80)fig, axes = plt.subplots(len(diag), 2, figsize=(14, 4 * len(diag)))if len(diag) == 1:    axes = np.array([axes])for r, window in enumerate(diag.keys()):    proba_tr = diag[window]["proba_train"]    proba_te = diag[window]["proba_test"]    df_tr = diag[window]["df_train"]    df_te = diag[window]["df_test"]    max_tr = proba_tr.max(axis=1)    max_te = proba_te.max(axis=1)    print(f"\n[{window}] certainty stats:")    print(f"  TRAIN mean max-proba={max_tr.mean():.3f} | ambiguous(<{AMBIGUOUS_THRESHOLD})={(max_tr < AMBIGUOUS_THRESHOLD).mean()*100:.1f}%")    print(f"  TEST  mean max-proba={max_te.mean():.3f} | ambiguous(<{AMBIGUOUS_THRESHOLD})={(max_te < AMBIGUOUS_THRESHOLD).mean()*100:.1f}%")    # histogram    ax = axes[r, 0]    ax.hist(max_tr, bins=50, alpha=0.6, label="train")    ax.hist(max_te, bins=50, alpha=0.6, label="test")    ax.axvline(AMBIGUOUS_THRESHOLD, color="red", linestyle="--", linewidth=2, label="threshold")    ax.set_title(f"{window} max posterior (hist)")    ax.grid(True, alpha=0.3)    ax.legend()    # time series    ax = axes[r, 1]    ax.plot(df_tr.index, max_tr, linewidth=0.6, alpha=0.7, label="train")    ax.plot(df_te.index, max_te, linewidth=0.6, alpha=0.7, label="test")    ax.axhline(AMBIGUOUS_THRESHOLD, color="red", linestyle="--", linewidth=2, label="threshold")    ax.axvline(df_tr.index.max(), color="black", linestyle="--", linewidth=1.5, label="split")    ax.set_title(f"{window} max posterior over time")    ax.grid(True, alpha=0.3)    ax.legend()plt.tight_layout()plt.savefig("hmm_3states_posterior_certainty.png", dpi=150, bbox_inches="tight")plt.close()print("\n✓ Saved: hmm_3states_posterior_certainty.png")# 7. STATE TIMELINE PLOTS (train/test concatenati SOLO per grafici)print("\n" + "=" * 80)print("7) STATE TIMELINES")print("=" * 80)for window in diag.keys():    df_tr = diag[window]["df_train"]    df_te = diag[window]["df_test"]    st_tr = diag[window]["states_train"]    st_te = diag[window]["states_test"]    all_idx = df_tr.index.append(df_te.index)    all_states = np.concatenate([st_tr, st_te])    split_date = df_tr.index.max()    fig, ax = plt.subplots(figsize=(14, 4))    ax.scatter(all_idx, all_states, c=all_states, cmap="viridis", s=6, alpha=0.7)    ax.axvline(split_date, color="red", linestyle="--", linewidth=2, label="Train/Test split")    ax.set_title(f"{window} - Hidden states timeline (decoded separately)")    ax.set_ylabel("State")    ax.grid(True, alpha=0.3)    ax.legend()    plt.tight_layout()    plt.savefig(f"hmm_states_timeline_{window}.png", dpi=150, bbox_inches="tight")    plt.close()    print(f"✓ Saved: hmm_states_timeline_{window}.png")# 8. HISTORICAL EVENT ALIGNMENT (ALL windows) + day-count discrepancyprint("\n" + "=" * 80)print("8) HISTORICAL EVENT ALIGNMENT (ALL windows) + DAY COUNT CHECK")print("=" * 80)all_rows = []for window in diag.keys():    feats = diag[window]["features"]    df_tr = diag[window]["df_train"]    df_te = diag[window]["df_test"]    st_tr = diag[window]["states_train"]    st_te = diag[window]["states_test"]    df_all = pd.concat([df_tr, df_te])    states_all = np.concatenate([st_tr, st_te])    # identifica "crisis state" dal VIX medio (sul TRAIN)    vix_feat = "VIX_lag1"    vix_idx = feats.index(vix_feat)    X_train = df_tr.values    vix_means = []    for s in range(N_STATES):        m = st_tr == s        vix_means.append(float(X_train[m, vix_idx].mean()) if m.any() else -np.inf)    crisis_state = int(np.argmax(vix_means))    print(f"\n[{window}] Crisis state (highest TRAIN mean VIX): S{crisis_state} | means={vix_means}")    crisis_mask_all = (states_all == crisis_state)    rows = []    for start, end, name in KNOWN_EVENTS:        start_dt = pd.to_datetime(start)        end_dt = pd.to_datetime(end)        calendar_days = int((end_dt - start_dt).days + 1)        # lun-ven (non calendario NYSE)        bdays = pd.bdate_range(start_dt, end_dt)        business_days_mon_fri = int(len(bdays))        # giorni effettivamente presenti nel tuo dataset (trading days)        in_dataset_mask = (df_all.index >= start_dt) & (df_all.index <= end_dt)        trading_days_in_dataset = int(in_dataset_mask.sum())        dataset_days = df_all.index[in_dataset_mask]        missing_vs_bdays = bdays.difference(dataset_days)        n_missing_vs_bdays = int(len(missing_vs_bdays))        if trading_days_in_dataset > 0:            crisis_days = int(crisis_mask_all[in_dataset_mask].sum())            crisis_pct = float(crisis_days / trading_days_in_dataset * 100.0)        else:            crisis_days = 0            crisis_pct = np.nan        row = {            "window": window,            "crisis_state": crisis_state,            "event": name,            "start": start,            "end": end,            "calendar_days": calendar_days,            "business_days_mon_fri": business_days_mon_fri,            "trading_days_in_dataset": trading_days_in_dataset,            "missing_vs_monfri": n_missing_vs_bdays,            "crisis_days_in_dataset": crisis_days,            "crisis_pct_of_dataset_days": crisis_pct        }        rows.append(row)        all_rows.append(row)    events_df = pd.DataFrame(rows)    out_csv = f"hmm_event_daycount_discrepancy_{window}.csv"    events_df.to_csv(out_csv, index=False)    print(f"✓ Saved: {out_csv}")# tabella unica riassuntiva (tutte le finestre insieme)events_all_df = pd.DataFrame(all_rows)events_all_df.to_csv("hmm_event_daycount_discrepancy_ALL.csv", index=False)print("✓ Saved: hmm_event_daycount_discrepancy_ALL.csv")

## Stage 7 — LSTM configuration and imports

In [ ]:
import gcimport warningswarnings.filterwarnings("ignore")import numpy as npimport pandas as pdimport tensorflow as tffrom tensorflow import kerasfrom tensorflow.keras import layers, regularizersfrom sklearn.model_selection import TimeSeriesSplitfrom sklearn.preprocessing import StandardScaler#  CONFIGWINDOWS     = ["20d", "60d", "120d"]TRAIN_PROP  = 0.8N_SPLITS    = 5QUANTILES   = [0.05, 0.25, 0.50, 0.75, 0.95]N_QUANTILES = len(QUANTILES)N_STATES    = 3RANDOM_STATE = 42SEEDS       = [42, 0, 7, 13, 99, 1, 2, 3, 5, 21]# Griglia iperparametriSEQ_LENGTH_GRID = [10, 20, 30]UNITS_GRID      = [8, 16, 32, 64]DROPOUT1_GRID   = [0.1, 0.2, 0.3, 0.4, 0.5]L2_GRID         = [0.001, 0.002, 0.003, 0.004, 0.005]LR_GRID         = [1e-3, 9e-4, 5e-4]# Ricerca random (Stage 1 veloce)USE_RANDOM_SEARCH  = TrueN_RANDOM_SAMPLES   = 40N_SPLITS_TUNING    = 3EPOCHS_TUNING      = 10# Refit finale e Stage 2TOP_K   = 20EPOCHS  = 20BATCH_SIZE = 64np.random.seed(RANDOM_STATE)tf.random.set_seed(RANDOM_STATE)

## Stage 8 — Load standardised dataset

In [ ]:
df_full = pd.read_csv("dataset_standardized.csv", index_col=0, parse_dates=True)df_full["Yield_Diff_change"]     = df_full["Yield_Diff_lag1"].diff()df_full["vol_trend_120d_change"] = df_full["vol_trend_120d"].diff()print(f"Dataset: {len(df_full)} righe | {df_full.index.min().date()} → {df_full.index.max().date()}")

## Stage 9 — Feature configuration per horizon

In [ ]:
# NOTA: ret_{w}d è ESCLUSO dalle feature LSTM.# Motivo: ret_20d = rolling sum di ret_1d su 20gg  autocorrelazione# strutturale ~1 tra osservazioni consecutive (overlap di 19gg su 20).# Includerlo come feature crea quasi-data-leakage nelle sequenze e gonfia# artificialmente le metriche. Si usa solo come TARGET.# Per la finestra 20d, la dinamica di breve periodo è già catturata da# VIX_lag1, skew_20d, kurt_20d e dalla sequenza stessa (seq_len step).BASE_FEATURES = {    "20d":  ["VIX_lag1", "vol_trend_20d",         "skew_20d",  "kurt_20d",             "oil_ret",  "gold_ret",   "dxy_ret",  "Yield_Diff_change",             "VIX_term_lag1", "VIX_sent_lag1"],    "60d":  ["VIX_lag1", "vol_trend_60d",          "skew_60d",  "kurt_60d",             "oil_ret",  "gold_ret",   "dxy_ret",  "Yield_Diff_change",             "VIX_term_lag1", "VIX_sent_lag1"],    "120d": ["VIX_lag1", "vol_trend_120d_change",  "skew_120d", "kurt_120d",             "oil_ret",  "gold_ret",   "dxy_ret",  "Yield_Diff_change",             "VIX_term_lag1", "VIX_sent_lag1"],}TARGET = {    "20d":  "ret_20d",    "60d":  "ret_60d",    "120d": "ret_120d",}

## Stage 10 — Load HMM posterior probabilities (soft regime inputs)

In [ ]:
def load_hmm_posteriors(window: str, n_states: int = N_STATES) -> pd.DataFrame:    """    Carica le probabilità posteriori P(S=k | obs) salvate durante il fitting HMM.    Restituisce un DataFrame con colonne [p_state_0, p_state_1, p_state_2]    indicizzato per data.    I file sono costruiti su train e test SEPARATI, rispettando il protocollo    walk-forward (nessun look-ahead del test nel fitting).    """    cols = [f"p_state_{i}" for i in range(n_states)]    pr_tr = pd.read_csv(        f"hmm_proba_train_{window}_3states.csv", index_col=0, parse_dates=True    )    pr_te = pd.read_csv(        f"hmm_proba_test_{window}_3states.csv",  index_col=0, parse_dates=True    )    # Rinomina colonne se necessario (i CSV hanno già p_state_0/1/2)    pr_tr.columns = cols    pr_te.columns = cols    return pd.concat([pr_tr, pr_te]).sort_index()def add_soft_states(df: pd.DataFrame, posteriors: pd.DataFrame) -> pd.DataFrame:    """    Unisce le probabilità posteriori al dataframe delle feature.    Le colonne p_state_0/1/2 sono continue in [0,1] e sommano a 1.    """    return df.join(posteriors, how="inner")

## Stage 11 — Pinball loss and LSTM helper functions

In [ ]:
def quantile_loss_tf(quantiles):    q = tf.constant(quantiles, dtype=tf.float32)    def loss(y_true, y_pred):        y_true = tf.expand_dims(y_true, -1)        e      = y_true - y_pred        return tf.reduce_mean(tf.maximum(q * e, (q - 1.0) * e))    return lossdef pinball_loss_np(y_true: np.ndarray, y_pred: np.ndarray, quantiles) -> float:    y_true = y_true.reshape(-1, 1)    losses = []    for j, q in enumerate(quantiles):        e = y_true[:, 0] - y_pred[:, j]        losses.append(np.mean(np.maximum(q * e, (q - 1) * e)))    return float(np.mean(losses))def make_sequences_all(df: pd.DataFrame, feature_cols, target_col: str, seq_len: int):    """Costruisce sequenze (X, y) in float32 senza loop Python visibile."""    X, y = [], []    vals_X = df[feature_cols].values.astype(np.float32)    vals_y = df[target_col].values.astype(np.float32)    for i in range(seq_len, len(df) - 1):        X.append(vals_X[i - seq_len : i])        y.append(vals_y[i + 1])    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)def fold_slice_by_row_indices(seq_len, n_rows, tr_idx, va_idx):    target_rows = np.arange(seq_len + 1, n_rows)    tr_seq = np.where(np.isin(target_rows, tr_idx))[0]    va_seq = np.where(np.isin(target_rows, va_idx))[0]    return tr_seq, va_seqdef build_model(seq_len, n_features, units, dropout1, l2, lr):    model = keras.Sequential([        layers.Input(shape=(seq_len, n_features)),        layers.LSTM(            int(units),            return_sequences=False,            kernel_regularizer=regularizers.l2(float(l2)),            recurrent_regularizer=regularizers.l2(float(l2)),        ),        layers.BatchNormalization(),        layers.Dropout(float(dropout1)),        layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(float(l2))),        layers.Dropout(float(dropout1)),        layers.Dense(16, activation="relu"),        layers.Dropout(0.2),        layers.Dense(N_QUANTILES, activation="linear"),    ])    model.compile(        optimizer=keras.optimizers.Adam(learning_rate=float(lr)),        loss=quantile_loss_tf(QUANTILES),    )    return modeldef full_param_grid():    for seq_len in SEQ_LENGTH_GRID:        for units in UNITS_GRID:            for dropout1 in DROPOUT1_GRID:                for l2 in L2_GRID:                    for lr in LR_GRID:                        yield {"seq_len": seq_len, "units": units,                               "dropout1": dropout1, "l2": l2, "lr": lr}def random_param_samples(n):    params_all = list(full_param_grid())    rng = np.random.default_rng(RANDOM_STATE)    idx = rng.choice(len(params_all), size=min(n, len(params_all)), replace=False)    for i in idx:        yield params_all[i]def fix_quantile_crossing(yhat: np.ndarray) -> np.ndarray:    """    yhat shape: (n, 5) — colonne ordinate per QUANTILES = [0.05, 0.25, 0.50, 0.75, 0.95]    Applica isotonic regression per forzare la monotonia riga per riga.    Restituisce array della stessa shape con q05 <= q25 <= q50 <= q75 <= q95.    """    from sklearn.isotonic import IsotonicRegression    ir = IsotonicRegression(increasing=True)    out = yhat.copy()    for i in range(len(yhat)):        out[i] = ir.fit_transform(np.arange(5), yhat[i])    return outdef check_and_fix_crossing(yhat: np.ndarray, quantiles, window: str) -> np.ndarray:    """    Controlla il crossing rate, logga, e applica fix isotonic.    Ritorna yhat corretto.    """    n = len(yhat)    crossing_mask = np.zeros(n, dtype=bool)    for j in range(len(quantiles) - 1):        crossing_mask |= (yhat[:, j] > yhat[:, j + 1])    rate = crossing_mask.mean()    print(f"  [{window}] quantile crossing: {crossing_mask.sum()}/{n} obs ({rate*100:.1f}%)", end="")    if rate > 0:        yhat = fix_quantile_crossing(yhat)        print(" → fixed ✓")    else:        print(" → nessun crossing")    return yhat

## Stage 12 — Random search with walk-forward cross-validation

In [ ]:
all_summary = []# Prima di partire con il loop WINDOWSgc.collect()tf.keras.backend.clear_session()# Controlla quanta RAM stai usando in quel momentoimport psutilused = psutil.virtual_memory().used / 1024**3total = psutil.virtual_memory().total / 1024**3print(f"RAM usata: {used:.1f} / {total:.1f} GB")for window in WINDOWS:    print("\n" + "="*100)    print(f"WINDOW {window} | Stage 1 – random search walk-forward CV (SOFT HMM posteriors)")    print("="*100)    target    = TARGET[window]    base_cols = BASE_FEATURES[window]    #  1. Carica probabilità posteriori HMM (soft, continue)    posteriors = load_hmm_posteriors(window)    soft_cols  = [f"p_state_{i}" for i in range(N_STATES)]    #  2. Costruisci dataframe unificato    df = df_full[base_cols + [target]].dropna().copy()    df = add_soft_states(df, posteriors)          # aggiunge p_state_0/1/2    feat_cols = base_cols + soft_cols             # 10 base + 3 soft = 13 feature totali    print(f"  Feature totali: {len(feat_cols)}  |  righe dopo join: {len(df)}")    print(f"  Colonne HMM (soft): {soft_cols}")    print(f"  Range p_state_0: [{df['p_state_0'].min():.3f}, {df['p_state_0'].max():.3f}]  "          f"(atteso continuo, non binario)")    #  3. Split train/test    split_idx = int(len(df) * TRAIN_PROP)    df_train  = df.iloc[:split_idx].copy()    df_test   = df.iloc[split_idx:].copy()    #  4. Pre-calcola sequenze per ogni seq_len (una volta sola)    seq_cache = {}    for seq_len in SEQ_LENGTH_GRID:        X_all, y_all = make_sequences_all(df_train, feat_cols, target, seq_len)        seq_cache[seq_len] = (X_all, y_all)        print(f"  cached seq_len={seq_len}: X={X_all.shape}  "              f"({X_all.nbytes/1024**2:.1f} MB)")    #  5. Random search    tscv = TimeSeriesSplit(n_splits=N_SPLITS_TUNING)    rows = []    for pi, p in enumerate(random_param_samples(N_RANDOM_SAMPLES), 1):        X_all, y_all = seq_cache[p["seq_len"]]        n_rows = len(df_train)        fold_losses = []        for tr_idx, va_idx in tscv.split(df_train):            tr_seq, va_seq = fold_slice_by_row_indices(p["seq_len"], n_rows, tr_idx, va_idx)            if len(tr_seq) == 0 or len(va_seq) == 0:                continue            X_tr, y_tr = X_all[tr_seq], y_all[tr_seq]            X_va, y_va = X_all[va_seq], y_all[va_seq]            sy = StandardScaler()            y_tr_s = sy.fit_transform(y_tr.reshape(-1, 1)).ravel()            tf.keras.backend.clear_session()            m = build_model(p["seq_len"], X_tr.shape[2], p["units"],                            p["dropout1"], p["l2"], p["lr"])            m.fit(X_tr, y_tr_s, epochs=EPOCHS_TUNING, batch_size=BATCH_SIZE, verbose=0)            yhat_va_s = m.predict(X_va, verbose=0)            yhat_va   = sy.inverse_transform(yhat_va_s)            fold_losses.append(pinball_loss_np(y_va, yhat_va, QUANTILES))            del m, sy, y_tr_s, yhat_va_s, yhat_va            gc.collect()        if not fold_losses:            continue        rows.append({**p,                     "n_folds_used":      len(fold_losses),                     "cv_pinball_mean":   float(np.mean(fold_losses)),                     "cv_pinball_std":    float(np.std(fold_losses))})        if pi % 10 == 0:            best_now = min(r["cv_pinball_mean"] for r in rows)            print(f"  [{window}] {pi}/{N_RANDOM_SAMPLES} configs | best={best_now:.6g}")    cv1 = pd.DataFrame(rows).sort_values(["cv_pinball_mean", "cv_pinball_std"])    cv1.to_csv(f"walkforward_quantile_cv_SOFT_{window}.csv", index=False)    print(f"\n  Best Stage1 ({window}): {cv1.iloc[0].to_dict()}")    del seq_cache    gc.collect()

## Stage 13 — Stage-2 refinement on the top-K configurations

In [ ]:
stage2_summary = []for window in WINDOWS:    print("\n" + "="*90)    print(f"STAGE 2 (TOP_K={TOP_K}, 5 fold) | {window} | SOFT HMM posteriors")    print("="*90)    target    = TARGET[window]    base_cols = BASE_FEATURES[window]    soft_cols = [f"p_state_{i}" for i in range(N_STATES)]    posteriors = load_hmm_posteriors(window)    df = df_full[base_cols + [target]].dropna().copy()    df = add_soft_states(df, posteriors)    feat_cols = base_cols + soft_cols    split_idx = int(len(df) * TRAIN_PROP)    df_train  = df.iloc[:split_idx].copy()    df_test   = df.iloc[split_idx:].copy()    # Carica top-K da Stage 1    cv1  = pd.read_csv(f"walkforward_quantile_cv_SOFT_{window}.csv")    cv1  = cv1.sort_values(["cv_pinball_mean", "cv_pinball_std"]).head(TOP_K)    needed_seq = sorted(cv1["seq_len"].astype(int).unique().tolist())    seq_cache  = {}    for sl in needed_seq:        X_all, y_all = make_sequences_all(df_train, feat_cols, target, sl)        seq_cache[sl] = (X_all, y_all)        print(f"  cached seq_len={sl}: {X_all.shape}  ({X_all.nbytes/1024**2:.1f} MB)")    tscv = TimeSeriesSplit(n_splits=N_SPLITS)    rows = []    for i, p in enumerate(cv1.to_dict("records"), 1):        sl    = int(p["seq_len"])        X_all, y_all = seq_cache[sl]        n_rows = len(df_train)        fold_losses = []        for tr_idx, va_idx in tscv.split(df_train):            tr_seq, va_seq = fold_slice_by_row_indices(sl, n_rows, tr_idx, va_idx)            if len(tr_seq) == 0 or len(va_seq) == 0:                continue            X_tr, y_tr = X_all[tr_seq], y_all[tr_seq]            X_va, y_va = X_all[va_seq], y_all[va_seq]            sy = StandardScaler()            y_tr_s = sy.fit_transform(y_tr.reshape(-1, 1)).ravel()            tf.keras.backend.clear_session()            m = build_model(sl, X_tr.shape[2], p["units"], p["dropout1"], p["l2"], p["lr"])            m.fit(X_tr, y_tr_s, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)            yhat_va_s = m.predict(X_va, verbose=0)            yhat_va   = sy.inverse_transform(yhat_va_s)            fold_losses.append(pinball_loss_np(y_va, yhat_va, QUANTILES))            del m, sy, y_tr_s, yhat_va_s, yhat_va            gc.collect()        rows.append({            "seq_len": sl, "units": int(p["units"]),            "dropout1": float(p["dropout1"]), "l2": float(p["l2"]), "lr": float(p["lr"]),            "n_folds_used":           len(fold_losses),            "stage2_cv_pinball_mean": float(np.mean(fold_losses)),            "stage2_cv_pinball_std":  float(np.std(fold_losses)),        })        if i % 5 == 0:            best_s2 = min(r["stage2_cv_pinball_mean"] for r in rows)            print(f"  {i}/{len(cv1)} configs | best={best_s2:.6g}")    cv2  = pd.DataFrame(rows).sort_values(["stage2_cv_pinball_mean", "stage2_cv_pinball_std"])    cv2.to_csv(f"walkforward_quantile_cv_stage2_SOFT_{window}.csv", index=False)    best = cv2.iloc[0].to_dict()    print(f"\n  BEST ({window}): {best}")    del seq_cache    gc.collect()

## Stage 14 — Final multi-seed ensemble (out-of-sample)Refits the best configuration across several random seeds and averages the predictions, saving the final out-of-sample quantile forecasts.

In [ ]:
import numpy as npimport pandas as pdfrom sklearn.preprocessing import StandardScalerimport tensorflow as tfimport gcSEEDS = [42, 0, 7, 13, 99, 1, 2, 3, 5, 21]all_run_results = {w: [] for w in WINDOWS}all_ensemble_pinballs_per_window = {} # New dictionary to store ensemble pinballsfor window in WINDOWS:    print(f"\n{'='*80}\nREFIT MULTI-SEED | {window}\n{'='*80}")    target    = TARGET[window]    base_cols = BASE_FEATURES[window]    soft_cols = [f"p_state_{i}" for i in range(N_STATES)]    posteriors = load_hmm_posteriors(window)    df = df_full[base_cols + [target]].dropna().copy()    df = add_soft_states(df, posteriors)    feat_cols = base_cols + soft_cols    split_idx = int(len(df) * TRAIN_PROP)    df_train  = df.iloc[:split_idx].copy()    df_test   = df.iloc[split_idx:].copy()    best = pd.read_csv(f"walkforward_quantile_cv_stage2_SOFT_{window}.csv").iloc[0].to_dict()    sl   = int(best["seq_len"])    # Sequenze train e test (identiche per tutti i seed)    X_tr, y_tr = make_sequences_all(df_train, feat_cols, target, sl)    X_te, y_te, d_te = [], [], []    for j in range(sl, len(df_test) - 1):        X_te.append(df_test[feat_cols].iloc[j - sl : j].values)        y_te.append(df_test[target].iloc[j + 1])        d_te.append(df_test.index[j + 1])    X_te = np.array(X_te, dtype=np.float32)    y_te = np.array(y_te, dtype=np.float32)    sy     = StandardScaler()    y_tr_s = sy.fit_transform(y_tr.reshape(-1, 1)).ravel()    all_yhats = []    seed_pinballs = []    for seed in SEEDS:        np.random.seed(seed)        tf.random.set_seed(seed)        tf.keras.backend.clear_session()        model = build_model(sl, X_tr.shape[2],                            best["units"], best["dropout1"], best["l2"], best["lr"])        model.fit(X_tr, y_tr_s, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)        yhat_s  = model.predict(X_te, verbose=0)        yhat    = sy.inverse_transform(yhat_s)        yhat = check_and_fix_crossing(yhat, QUANTILES, window)        all_yhats.append(yhat.copy())        pb      = pinball_loss_np(y_te, yhat, QUANTILES)        seed_pinballs.append(pb)        all_run_results[window].append(pb)        print(f"  seed={seed:>3} | pinball={pb:.6f}")        # Tieni il modello del seed migliore        del model, yhat_s, yhat        gc.collect()    run_arr = np.array(all_run_results[window])    print(f"\n  [{window}] pinball: mean={run_arr.mean():.6f} \u00b1 std={run_arr.std():.6f}")    ensemble_yhat = np.mean(all_yhats, axis=0)    ensemble_pb = pinball_loss_np(y_te, ensemble_yhat, QUANTILES)    all_ensemble_pinballs_per_window[window] = ensemble_pb # Store ensemble_pb for current window    # Best seed solo come sensitivity check (NON usato per salvare il CSV)    best_seed = SEEDS[np.argmin(seed_pinballs)]    best_pinball = np.min(seed_pinballs)    print(f" [{window}] pinball ensemble mean = {ensemble_pb:.6f}")    print(f" [{window}] best seed={best_seed} | best pinball={best_pinball:.6f} [solo riferimento]")    # Salva predizioni ensemble (NON del best seed)    out = pd.DataFrame(ensemble_yhat, columns=[f"q{int(q*100):02d}" for q in QUANTILES])    out.insert(0, "date", d_te)    out["y_true"] = y_te    out.to_csv(f"final_quantile_predictions_SOFT_{window}.csv", index=False)    print(f" \u2192 Salvato: final_quantile_predictions_SOFT_{window}.csv (ensemble di {len(SEEDS)} seed)")    del X_tr, y_tr, X_te, y_te    gc.collect()# Riepilogo finaleprint(f"\n{'='*80}")print("RIEPILOGO MULTI-SEED")print(f"{'='*80}")summary_rows = []for window in WINDOWS:    arr = np.array(all_run_results[window])    summary_rows.append({        "window":      window,        "pinball_mean": arr.mean(),        "pinball_std":  arr.std(),        "pinball_min":  arr.min(),        "pinball_max":  arr.max(),        "best_seed": SEEDS[np.argmin(arr)],   # solo riferimento        "ensemble_pinball": all_ensemble_pinballs_per_window[window] # Use stored value    })    print(f"  {window}: {arr.mean():.6f} \u00b1 {arr.std():.6f}  "          f"[min={arr.min():.6f}, max={arr.max():.6f}]")pd.DataFrame(summary_rows).to_csv("multiseed_summary_SOFT.csv", index=False)print("\n\u2713 Salvato: multiseed_summary_SOFT.csv")

## Stage 15 — Coverage and calibration metricsPICP, interval width, Winkler scores and posterior entropy.

In [ ]:
import pandas as pd, numpy as npQUANTILES = [0.05, 0.25, 0.50, 0.75, 0.95]def pinball(y, yhat, q):    e = y - yhat    return np.mean(np.maximum(q*e, (q-1)*e))rows = []for w in ["20d","60d","120d"]:    df = pd.read_csv(f"final_quantile_predictions_SOFT_{w}.csv")    y   = df["y_true"].values    q05, q25, q50, q75, q95 = [df[f"q{int(q*100):02d}"].values for q in QUANTILES]    pb   = np.mean([pinball(y, df[f"q{int(q*100):02d}"].values, q) for q in QUANTILES])    picp90 = np.mean((y >= q05) & (y <= q95))    picp50 = np.mean((y >= q25) & (y <= q75))    width90 = np.mean(q95 - q05)    rows.append({"window": w, "pinball": pb, "PICP90": picp90,                 "PICP50": picp50, "width90": width90,                 "below_q05": np.mean(y < q05), "above_q95": np.mean(y > q95)})print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
for w in ["20d","60d","120d"]:    p = pd.read_csv(f"hmm_proba_train_{w}_3states.csv", index_col=0).values    entropy = -np.sum(p * np.log(p + 1e-10), axis=1)    print(f"{w}: entropia media={entropy.mean():.3f} | % obs con max_p > 0.95: "          f"{(p.max(axis=1) > 0.95).mean()*100:.1f}%")

In [ ]:
# Aggiunge a PICP90/50 già calcolati:#   1. PICP per ogni singolo quantile (copertura marginale)#   2. Winkler Score per gli intervalli [q05,q95] e [q25,q75]#   3. Unconditional Coverage test di Kupiec per q05 e q95# Input:  final_quantile_predictions_SOFT_{w}.csv  (già prodotti da STAGE 8.2)# Output: calibration_metrics_full.csvimport numpy as npimport pandas as pdfrom scipy import stats#  1. Winkler Scoredef winkler_score(y: np.ndarray, lower: np.ndarray,                  upper: np.ndarray, alpha: float) -> float:    """    Winkler (1972): penalizza ampiezza + escursioni fuori intervallo.    alpha = livello di copertura nominale (0.90 per [q05,q95], 0.50 per [q25,q75])    WS = (upper - lower) + (2/alpha) * max(lower-y, 0) + (2/alpha) * max(y-upper, 0)    """    width = upper - lower    pen_below = (2.0 / alpha) * np.maximum(lower - y, 0.0)    pen_above = (2.0 / alpha) * np.maximum(y - upper, 0.0)    return float(np.mean(width + pen_below + pen_above))#  2. Kupiec Unconditional Coverage testdef kupiec_test(y: np.ndarray, q_hat: np.ndarray,                tau: float) -> dict:    """    H0: P(y < q_hat) = tau  (copertura corretta per il quantile tau)    Statistica LR = -2 * [log L(tau) - log L(tau_hat)]    con tau_hat = n_violations / n, distribuzione asintotica chi2(1).    Per q05 e q95 (VaR coverage test):      - violation  per q05: y < q05  (miss sul lato sinistro)      - violation  per q95: y > q95  (miss sul lato destro)    """    n = len(y)    if tau <= 0.5:        violations = np.sum(y < q_hat)    else:        violations = np.sum(y > q_hat)    tau_hat = violations / n    tau_hat = np.clip(tau_hat, 1e-10, 1 - 1e-10)    tau_c   = np.clip(tau,     1e-10, 1 - 1e-10)    # Log-likelihood ratio    lr = -2.0 * (        violations * np.log(tau_c / tau_hat)        + (n - violations) * np.log((1 - tau_c) / (1 - tau_hat))    )    p_value = float(1 - stats.chi2.cdf(lr, df=1))    return {        "n":           n,        "violations":  int(violations),        "tau_hat":     round(tau_hat, 4),        "tau_nominal": tau,        "LR_stat":     round(float(lr), 4),        "p_value":     round(p_value, 4),        "reject_H0":   p_value < 0.05,   # True → copertura non corretta    }#  3. Loop principaleQUANTILES = [0.05, 0.25, 0.50, 0.75, 0.95]q_cols    = {q: f"q{int(q*100):02d}" for q in QUANTILES}rows_full   = []   # una riga per (window, metrica)rows_kupiec = []   # una riga per (window, quantile)for w in ["20d", "60d", "120d"]:    df = pd.read_csv(f"final_quantile_predictions_SOFT_{w}.csv")    y  = df["y_true"].values    preds = {q: df[q_cols[q]].values for q in QUANTILES}    print(f"\n{'='*65}")    print(f"  CALIBRAZIONE COMPLETA | {w}  (n={len(y)})")    print(f"{'='*65}")    #  3a. PICP marginale per ogni quantile    print(f"\n  PICP marginale (nominale → realizzato):")    picp_row = {"window": w}    for q in QUANTILES:        if q <= 0.5:            picp = float(np.mean(y <= preds[q]))   # P(Y <= q_hat) ≈ q        else:            picp = float(np.mean(y <= preds[q]))        picp_row[f"picp_{q_cols[q]}"] = round(picp, 4)        print(f"    q{int(q*100):02d}: nominale={q:.2f}  realizzato={picp:.4f}  "              f"  bias={picp - q:+.4f}")    #  3b. PICP intervalli (già in STAGE 9, qui per completezza)    picp90 = float(np.mean((y >= preds[0.05]) & (y <= preds[0.95])))    picp50 = float(np.mean((y >= preds[0.25]) & (y <= preds[0.75])))    picp_row["picp90"] = round(picp90, 4)    picp_row["picp50"] = round(picp50, 4)    print(f"\n  PICP intervalli:")    print(f"    [q05,q95] PICP90: {picp90:.4f}  (nominale=0.90)")    print(f"    [q25,q75] PICP50: {picp50:.4f}  (nominale=0.50)")    #  3c. Winkler Score    ws90 = winkler_score(y, preds[0.05], preds[0.95], alpha=0.90)    ws50 = winkler_score(y, preds[0.25], preds[0.75], alpha=0.50)    width90 = float(np.mean(preds[0.95] - preds[0.05]))    width50 = float(np.mean(preds[0.75] - preds[0.25]))    picp_row["winkler90"] = round(ws90, 6)    picp_row["winkler50"] = round(ws50, 6)    picp_row["width90"]   = round(width90, 6)    picp_row["width50"]   = round(width50, 6)    print(f"\n  Winkler Score (minore = meglio):")    print(f"    [q05,q95] WS90={ws90:.6f}  width90={width90:.6f}")    print(f"    [q25,q75] WS50={ws50:.6f}  width50={width50:.6f}")    rows_full.append(picp_row)    #  3d. Kupiec test su q05 e q95 (VaR tail coverage)    print(f"\n  Kupiec UC test (H0: copertura corretta):")    for q in [0.05, 0.95]:        k = kupiec_test(y, preds[q], tau=q)        status = "❌ RIFIUTO H0" if k["reject_H0"] else "✅ non rifiuto"        print(f"    q{int(q*100):02d}: tau_hat={k['tau_hat']:.4f} "              f"(nominale={q:.2f}) | "              f"LR={k['LR_stat']:.3f} | p={k['p_value']:.4f} | {status}")        rows_kupiec.append({"window": w, "quantile": q, **k})#  4. Salvadf_calib  = pd.DataFrame(rows_full)df_kupiec = pd.DataFrame(rows_kupiec)df_calib.to_csv("calibration_metrics_full.csv",    index=False)df_kupiec.to_csv("calibration_kupiec_tests.csv",    index=False)print(f"\n\n{'='*65}")print("RIEPILOGO CALIBRAZIONE — tutte le finestre")print(f"{'='*65}")print(df_calib[["window", "picp90", "picp50",                "winkler90", "winkler50",                "width90", "width50"]].to_string(index=False))print(f"\n✓ Salvati: calibration_metrics_full.csv | calibration_kupiec_tests.csv")

## Stage 16 — Baselines: historical quantiles and GARCHHistorical-quantile and GARCH(1,1) (normal and Student-t) multi-step quantile forecasts used as benchmarks.

In [ ]:
import numpy as npimport pandas as pdfrom scipy import statsfrom arch import arch_modeldef pinball_baseline(y, yhat, q):    e = y - yhat    return np.mean(np.maximum(q * e, (q - 1) * e))def compute_metrics(y, preds_dict):    q05 = preds_dict[0.05]    q25 = preds_dict[0.25]    q75 = preds_dict[0.75]    q95 = preds_dict[0.95]    pb  = np.mean([pinball_baseline(y, preds_dict[q], q) for q in QUANTILES])    return {        "pinball":   pb,        "PICP90":    np.mean((y >= q05) & (y <= q95)),        "PICP50":    np.mean((y >= q25) & (y <= q75)),        "width90":   np.mean(q95 - q05),        "below_q05": np.mean(y < q05),        "above_q95": np.mean(y > q95),    }rows_hq     = []rows_garch  = []rows_garcht = []def garch_quantile_multistep(y_window_scaled, h, quantiles, n_sim=2000, dist="normal"):    """    Fitta GARCH(1,1) su y_window_scaled (in %, scala 100x),    simula n_sim traiettorie di h passi e restituisce i quantili    del ritorno cumulato (in scala originale, diviso 100).    dist: "normal" o "t"    """    res = arch_model(y_window_scaled, vol="Garch", p=1, q=1,                     dist=dist, mean="Constant").fit(disp="off", show_warning=False)    mu    = res.params["mu"]           # in scala %    omega = res.params["omega"]    alpha = res.params["alpha[1]"]    beta  = res.params["beta[1]"]    nu    = float(res.params.get("nu", 8.0))  # solo per dist="t"    # Varianza condizionale all'ultimo step osservato    last_var = float(res.conditional_volatility[-1] ** 2)    last_eps = float(res.resid[-1])    rng = np.random.default_rng(42)    if dist == "normal":        z = rng.standard_normal((n_sim, h))    else:        # t standardizzata: varianza = nu/(nu-2)  serve dividere per std        z_raw = rng.standard_t(df=nu, size=(n_sim, h))        z = z_raw / np.sqrt(nu / (nu - 2))    # Simula h passi di GARCH(1,1)    sim_rets = np.zeros((n_sim, h))    var_t = np.full(n_sim, last_var)    for t in range(h):        eps_prev = last_eps if t == 0 else sim_rets[:, t-1] - mu        var_t = omega + alpha * eps_prev**2 + beta * var_t        var_t = np.maximum(var_t, 1e-8)        sim_rets[:, t] = mu + np.sqrt(var_t) * z[:, t]    # Ritorno cumulato su h passi (in scala %)  converti in decimale    cum_ret = np.sum(sim_rets, axis=1) / 100.0  # appross. lineare, coerente col target    q_vals = {q: np.quantile(cum_ret, q) for q in quantiles}    return q_valsfor window in WINDOWS:    print(f"\n{'='*60}\nBASELINE | {window}\n{'='*60}")    target_col = TARGET[window]    df_pred = pd.read_csv(f"final_quantile_predictions_SOFT_{window}.csv",                          parse_dates=["date"])    y_te   = df_pred["y_true"].values    n_test = len(y_te)    y_all     = df_full[target_col].dropna().values    split_idx = int(len(y_all) * TRAIN_PROP)    y_train   = y_all[:split_idx]    print(f"  Train: {len(y_train)} obs | Test: {n_test} obs")    #  BASELINE 1: Historical Quantile    hq_preds = {q: np.full(n_test, np.quantile(y_train, q)) for q in QUANTILES}    m_hq = compute_metrics(y_te, hq_preds)    print(f"  HQ       | pinball={m_hq['pinball']:.6f} | "          f"PICP90={m_hq['PICP90']:.3f} | PICP50={m_hq['PICP50']:.3f} | "          f"width90={m_hq['width90']:.4f}")    rows_hq.append({"window": window, **m_hq})    #  SOSTITUISCI CON — rolling multi-step corretto    h = {"20d": 20, "60d": 60, "120d": 120}[window]    y_full_s = y_all * 100 if np.abs(y_all).max() < 1 else y_all    garch_n_q_list = {q: [] for q in QUANTILES}    garch_t_q_list = {q: [] for q in QUANTILES}    for i in range(n_test):        y_window = y_full_s[:split_idx + i]        q_n = garch_quantile_multistep(y_window, h, QUANTILES, n_sim=500, dist="normal")        q_t = garch_quantile_multistep(y_window, h, QUANTILES, n_sim=500, dist="t")        for q in QUANTILES:            garch_n_q_list[q].append(q_n[q])            garch_t_q_list[q].append(q_t[q])        if i % 100 == 0:            print(f" GARCH multi-step rolling: {i}/{n_test}", end="\r")    print()    garch_n_preds = {q: np.array(garch_n_q_list[q]) for q in QUANTILES}    garch_t_preds = {q: np.array(garch_t_q_list[q]) for q in QUANTILES}    # Save GARCH-N predictions with dates for Stage 11    df_garch_n_preds_out = pd.DataFrame({        "date": df_pred["date"].values,        "q05": garch_n_preds[0.05]    })    df_garch_n_preds_out.to_csv(f"baseline_garch_predictions_{window}.csv", index=False)    print(f"  → Salvato: baseline_garch_predictions_{window}.csv")    m_garch_n = compute_metrics(y_te, garch_n_preds)    m_garch_t = compute_metrics(y_te, garch_t_preds)    print(f"  GARCH-N  | pinball={m_garch_n['pinball']:.6f} | "          f"PICP90={m_garch_n['PICP90']:.3f} | PICP50={m_garch_n['PICP50']:.3f} | "          f"width90={m_garch_n['width90']:.4f}")    print(f"  GARCH-t  | pinball={m_garch_t['pinball']:.6f} | "          f"PICP90={m_garch_t['PICP90']:.3f} | PICP50={m_garch_t['PICP50']:.3f} | "          f"width90={m_garch_t['width90']:.4f}")    rows_garch.append({"window": window,  **m_garch_n})    rows_garcht.append({"window": window, **m_garch_t})# Salva tuttopd.DataFrame(rows_hq).to_csv("baseline_HQ_metrics.csv",      index=False)pd.DataFrame(rows_garch).to_csv("baseline_GARCH_metrics.csv",  index=False)pd.DataFrame(rows_garcht).to_csv("baseline_GARCHt_metrics.csv", index=False)#  RIEPILOGO CONFRONTO FINALEprint(f"\n{'='*80}")print("CONFRONTO: HMM-LSTM vs HQ vs GARCH-N vs GARCH-t")print(f"{'='*80}")df_lstm = pd.DataFrame([    {"window": "20d",  "pinball": 0.163541, "PICP90": 0.853231,     "PICP50": 0.447974, "width90": 1.914099},    {"window": "60d",  "pinball": 0.175644, "PICP90": 0.746479,     "PICP50": 0.366197, "width90": 1.440974},    {"window": "120d", "pinball": 0.180734, "PICP90": 0.618635,     "PICP50": 0.264355, "width90": 1.275375},])df_hq     = pd.DataFrame(rows_hq)df_garch  = pd.DataFrame(rows_garch)df_garcht = pd.DataFrame(rows_garcht)for w in WINDOWS:    lstm  = df_lstm[df_lstm.window == w].iloc[0]    hq    = df_hq[df_hq.window == w].iloc[0]    gn    = df_garch[df_garch.window == w].iloc[0]    gt    = df_garcht[df_garcht.window == w].iloc[0]    print(f"\n  {w}:")    print(f"    {'Modello':<12} {'Pinball':>9} {'PICP90':>8} {'PICP50':>8} {'Width90':>9}")    print(f"    {'─'*52}")    for label, row in [("HMM-LSTM", lstm), ("GARCH-t", gt),                        ("GARCH-N", gn),   ("HQ", hq)]:        print(f"    {label:<12} {row.pinball:>9.6f} {row.PICP90:>8.3f} "              f"{row.PICP50:>8.3f} {row.width90:>9.4f}")    print(f"    → LSTM vs GARCH-t: {(lstm.pinball-gt.pinball)/gt.pinball*100:+.1f}% | "          f"LSTM vs GARCH-N: {(lstm.pinball-gn.pinball)/gn.pinball*100:+.1f}% | "          f"LSTM vs HQ: {(lstm.pinball-hq.pinball)/hq.pinball*100:+.1f}%")print("\n✓ Salvati: baseline_HQ_metrics.csv, baseline_GARCH_metrics.csv, baseline_GARCHt_metrics.csv")

## Stage 17 — Systematic comparison and Diebold-Mariano test

In [ ]:
import numpy as npimport pandas as pdfrom scipy import statsQUANTILES = [0.05, 0.25, 0.50, 0.75, 0.95]WINDOWS   = ["20d", "60d", "120d"]def winkler_score(y, q_lo, q_hi, alpha):    """    Winkler Score per un intervallo a livello (1-alpha).    alpha = 0.10 per l'intervallo [q05, q95].    alpha = 0.50 per l'intervallo [q25, q75].    Più basso = meglio.    """    width = q_hi - q_lo    below = y < q_lo    above = y > q_hi    penalty = np.where(below, 2/alpha * (q_lo - y),              np.where(above, 2/alpha * (y - q_hi), 0.0))    return float(np.mean(width + penalty))def kupiec_test(y, q_hat, tau):    """    Kupiec POF test.    H0: P(Y < q_tau) = tau    """    n = len(y)    hits = np.sum(y < q_hat)    tau_hat = hits / n    if tau_hat == 0 or tau_hat == 1:        return np.nan, np.nan    lr = 2 * (hits * np.log(tau_hat / tau) +              (n - hits) * np.log((1 - tau_hat) / (1 - tau)))    p_val = 1 - stats.chi2.cdf(lr, df=1)    return float(lr), float(p_val)def dm_test(y, pred_a, pred_b, quantiles):    """    Diebold-Mariano test, one-sided:    H1: pred_a migliore di pred_b in termini di pinball loss.    """    def pinball_pointwise(y, yhat, q):        e = y - yhat        return np.maximum(q * e, (q - 1) * e)    d = np.zeros(len(y))    for q in quantiles:        col = f"q{int(q*100):02d}"        la = pinball_pointwise(y, pred_a[col].values, q)        lb = pinball_pointwise(y, pred_b[col].values, q)        d += (lb - la)    d /= len(quantiles)    n = len(d)    d_bar = np.mean(d)    h = max(1, int(n ** (1/3)))    gamma0 = np.var(d, ddof=1)    nw_var = gamma0    for k in range(1, h + 1):        gamma_k = np.cov(d[k:], d[:-k])[0, 1]        nw_var += 2 * (1 - k / (h + 1)) * gamma_k    nw_var = max(nw_var, 1e-10)    dm_stat = d_bar / np.sqrt(nw_var / n)    p_val = 1 - stats.norm.cdf(dm_stat)    return float(dm_stat), float(p_val)def build_hq_preds_df(y_train, n_test, quantiles):    d = {f"q{int(q*100):02d}": np.full(n_test, np.quantile(y_train, q))         for q in quantiles}    return pd.DataFrame(d)def build_garcht_preds_df(y_all, split_idx, n_test, quantiles, window_label):    """    Carica predizioni GARCH-t già salvate se esistono, altrimenti le ricalcola.    """    from arch import arch_model    fname = f"baseline_garcht_predictions_{window_label}.csv" # Changed 'preds' to 'predictions'    try:        df = pd.read_csv(fname)        print(f"  GARCH-t preds caricato da {fname}")        return df    except FileNotFoundError:        print(f"  Ricalcolo GARCH-t per {window_label} (può richiedere ~5 min)...")        y_s = y_all * 100 if np.abs(y_all).max() < 5 else y_all        res0 = arch_model(y_s[:split_idx], vol="Garch", p=1, q=1,                          dist="t", mean="Constant").fit(disp="off")        mu = res0.params["mu"] / 100        nu = float(res0.params.get("nu", 8.0))        fv = []        for i in range(n_test):            yw = y_s[:split_idx + i]            ri = arch_model(yw, vol="Garch", p=1, q=1,                            dist="t", mean="Constant").fit(                                disp="off", show_warning=False)            fc = ri.forecast(horizon=1)            fv.append(fc.variance.values[-1, 0])            nu = float(ri.params.get("nu", nu))        sigma = np.sqrt(np.array(fv)) / 100        d = {f"q{int(q*100):02d}": mu + sigma * stats.t.ppf(q, df=nu)             for q in quantiles}        df = pd.DataFrame(d)        df.to_csv(fname, index=False)        return dfrows_extended = []for window in WINDOWS:    print(f"\n{'='*70}")    print(f"STAGE 10b | {window}")    print(f"{'='*70}")    target_col = TARGET[window]    df_lstm_pred = pd.read_csv(        f"final_quantile_predictions_SOFT_{window}.csv",        parse_dates=["date"]    )    y_te = df_lstm_pred["y_true"].values    n_test = len(y_te)    y_all = df_full[target_col].dropna().values    split_i = int(len(y_all) * TRAIN_PROP)    y_train = y_all[:split_i]    df_hq_pred = build_hq_preds_df(y_train, n_test, QUANTILES)    df_hq_pred.to_csv(f"baseline_HQ_predictions_{window}.csv", index=False) # Added save for HQ predictions    df_gt_pred = build_garcht_preds_df(y_all, split_i, n_test, QUANTILES, window)    # The build_garcht_preds_df function now saves the file with the correct name internally, so no redundant save here.    q05_lstm = df_lstm_pred["q05"].values    q25_lstm = df_lstm_pred["q25"].values    q75_lstm = df_lstm_pred["q75"].values    q95_lstm = df_lstm_pred["q95"].values    q05_hq = df_hq_pred["q05"].values    q25_hq = df_hq_pred["q25"].values    q75_hq = df_hq_pred["q75"].values    q95_hq = df_hq_pred["q95"].values    q05_gt = df_gt_pred["q05"].values    q25_gt = df_gt_pred["q25"].values    q75_gt = df_gt_pred["q75"].values    q95_gt = df_gt_pred["q95"].values    ws90_lstm = winkler_score(y_te, q05_lstm, q95_lstm, alpha=0.10)    ws50_lstm = winkler_score(y_te, q25_lstm, q75_lstm, alpha=0.50)    ws90_hq   = winkler_score(y_te, q05_hq,   q95_hq,   alpha=0.10)    ws50_hq   = winkler_score(y_te, q25_hq,   q75_hq,   alpha=0.50)    ws90_gt   = winkler_score(y_te, q05_gt,   q95_gt,   alpha=0.10)    ws50_gt   = winkler_score(y_te, q25_gt,   q75_gt,   alpha=0.50)    print(f"  Winkler90  | LSTM={ws90_lstm:.4f} | GARCH-t={ws90_gt:.4f} | HQ={ws90_hq:.4f}")    print(f"  Winkler50  | LSTM={ws50_lstm:.4f} | GARCH-t={ws50_gt:.4f} | HQ={ws50_hq:.4f}")    lr_lstm, p_lstm = kupiec_test(y_te, q05_lstm, tau=0.05)    lr_hq,   p_hq   = kupiec_test(y_te, q05_hq,   tau=0.05)    lr_gt,   p_gt   = kupiec_test(y_te, q05_gt,   tau=0.05)    print(f"  Kupiec q05 | LSTM: LR={lr_lstm:.3f} p={p_lstm:.3f}"          f" | GARCH-t: LR={lr_gt:.3f} p={p_gt:.3f}"          f" | HQ: LR={lr_hq:.3f} p={p_hq:.3f}")    dm_vs_hq_stat, dm_vs_hq_p = dm_test(y_te, df_lstm_pred, df_hq_pred, QUANTILES)    dm_vs_gt_stat, dm_vs_gt_p = dm_test(y_te, df_lstm_pred, df_gt_pred, QUANTILES)    print(f"  DM test    | LSTM>HQ:      DM={dm_vs_hq_stat:+.3f}  p={dm_vs_hq_p:.3f}"          f"  {'✓ sign.' if dm_vs_hq_p < 0.05 else 'n.s.'}")    print(f"  DM test    | LSTM>GARCH-t: DM={dm_vs_gt_stat:+.3f}  p={dm_vs_gt_p:.3f}"          f"  {'✓ sign.' if dm_vs_gt_p < 0.05 else 'n.s.'}")    for model, ws90, ws50, lr_k, p_k, dm_stat, dm_p in [        ("HMM-LSTM", ws90_lstm, ws50_lstm, lr_lstm, p_lstm, None, None),        ("GARCH-t",  ws90_gt,   ws50_gt,   lr_gt,   p_gt,   dm_vs_gt_stat, dm_vs_gt_p),        ("HQ",       ws90_hq,   ws50_hq,   lr_hq,   p_hq,   dm_vs_hq_stat, dm_vs_hq_p),    ]:        rows_extended.append({            "window": window,            "model": model,            "winkler90": round(ws90, 4),            "winkler50": round(ws50, 4),            "kupiec_LR": round(lr_k, 3) if lr_k is not None else None,            "kupiec_p": round(p_k, 3) if p_k is not None else None,            "DM_vs_LSTM": round(dm_stat, 3) if dm_stat is not None else "—",            "DM_p": round(dm_p, 3) if dm_p is not None else "—",        })df_ext = pd.DataFrame(rows_extended)df_ext.to_csv("metrics_extended_2b.csv", index=False)print(f"\n{'='*70}")print("TABELLA RIEPILOGATIVA — PRIORITÀ 2b")print(f"{'='*70}")print(df_ext.to_string(index=False))print("\n✓ Salvato: metrics_extended_2b.csv")

## Stage 18 — Formal calibration tests (Christoffersen, Berkowitz)

In [ ]:
# Christoffersen (1998) + Berkowitz PIT testimport numpy as npimport pandas as pdfrom scipy import statsfrom scipy.special import ndtriQUANTILES = [0.05, 0.25, 0.50, 0.75, 0.95]WINDOWS   = ["20d", "60d", "120d"]def christoffersen_test(y, q_hat, tau):    hits  = (y < q_hat).astype(int)    n     = len(hits)    n1    = hits.sum()    n0    = n - n1    pihat = n1 / n    if pihat == 0 or pihat == 1:        lr_uc, p_uc = np.nan, np.nan    else:        lr_uc = 2 * (n1 * np.log(pihat / tau) +                     n0 * np.log((1 - pihat) / (1 - tau)))        p_uc  = 1 - stats.chi2.cdf(lr_uc, df=1)    n00 = ((hits[:-1] == 0) & (hits[1:] == 0)).sum()    n01 = ((hits[:-1] == 0) & (hits[1:] == 1)).sum()    n10 = ((hits[:-1] == 1) & (hits[1:] == 0)).sum()    n11 = ((hits[:-1] == 1) & (hits[1:] == 1)).sum()    pi01 = n01 / (n00 + n01) if (n00 + n01) > 0 else 0.0    pi11 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0.0    pi2  = (n01 + n11) / (n00 + n01 + n10 + n11)    def safe_log(x): return np.log(x) if x > 0 else 0.0    lr_ind = 2 * (        n00 * safe_log(1 - pi01) + n01 * safe_log(pi01) +        n10 * safe_log(1 - pi11) + n11 * safe_log(pi11) -        (n00 + n10) * safe_log(1 - pi2) -        (n01 + n11) * safe_log(pi2)    )    p_ind = 1 - stats.chi2.cdf(lr_ind, df=1)    lr_cc = lr_uc + lr_ind    p_cc  = 1 - stats.chi2.cdf(lr_cc, df=2)    return {        "lr_uc": round(float(lr_uc), 3),  "p_uc":  round(float(p_uc),  3),        "lr_ind": round(float(lr_ind), 3), "p_ind": round(float(p_ind), 3),        "lr_cc": round(float(lr_cc), 3),  "p_cc":  round(float(p_cc),  3),        "pi_hat": round(float(pihat), 4), "tau_nom": tau,        "n00": int(n00), "n01": int(n01), "n10": int(n10), "n11": int(n11),    }def pit_test(y, quantile_preds, quantiles):    q_arr = np.array(quantiles)    n     = len(y)    u     = np.zeros(n)    q_mat = quantile_preds.values    for i in range(n):        qi = q_mat[i]        yi = y[i]        if yi <= qi[0]:            u[i] = max(0.001, q_arr[0] * yi / qi[0]) if qi[0] != 0 else 0.001        elif yi >= qi[-1]:            u[i] = 0.999        else:            j    = np.searchsorted(qi, yi) - 1            j    = max(0, min(j, len(qi) - 2))            frac = (yi - qi[j]) / (qi[j+1] - qi[j] + 1e-10)            u[i] = q_arr[j] + frac * (q_arr[j+1] - q_arr[j])        u[i] = np.clip(u[i], 0.001, 0.999)    ks_stat, ks_p = stats.kstest(u, "uniform")    z      = np.clip(ndtri(u), -6, 6)    z_lag  = z[:-1]    z_cur  = z[1:]    X      = np.column_stack([np.ones(len(z_lag)), z_lag])    beta, *_ = np.linalg.lstsq(X, z_cur, rcond=None)    mu_hat, rho_hat = beta    resid   = z_cur - X @ beta    sigma2  = np.var(resid, ddof=2)    n_z     = len(z_cur)    ll_r    = -n_z/2 * np.log(2*np.pi) - np.sum(z_cur**2) / 2    ll_u    = -n_z/2 * np.log(2*np.pi*sigma2) - n_z/2    lr_berk = 2 * (ll_u - ll_r)    p_berk  = 1 - stats.chi2.cdf(lr_berk, df=3)    return {        "ks_stat": round(float(ks_stat), 4), "ks_p":    round(float(ks_p),    3),        "berk_lr": round(float(lr_berk), 3), "berk_p":  round(float(p_berk),  3),        "rho_hat": round(float(rho_hat), 3), "mu_hat":  round(float(mu_hat),  4),    }rows_cc  = []rows_pit = []TAUS_TO_TEST = [0.05, 0.25, 0.75, 0.95]for window in WINDOWS:    print(f"\n{'='*70}")    print(f"STAGE 10c | {window}")    print(f"{'='*70}")    df_pred = pd.read_csv(f"final_quantile_predictions_SOFT_{window}.csv",                          parse_dates=["date"])    y_te = df_pred["y_true"].values    print(f"\n  {'tau':>6}  {'π̂':>7}  {'LR_uc':>7}  {'p_uc':>6}  "          f"{'LR_ind':>7}  {'p_ind':>6}  {'LR_cc':>7}  {'p_cc':>6}  Esito")    print(f"  {'-'*68}")    for tau in TAUS_TO_TEST:        col   = f"q{int(tau*100):02d}"        q_hat = df_pred[col].values        cc    = christoffersen_test(y_te, q_hat, tau)        esito = f"UC{'✓' if cc['p_uc']>=0.05 else '✗'} IND{'✓' if cc['p_ind']>=0.05 else '✗'} CC{'✓' if cc['p_cc']>=0.05 else '✗'}"        print(f"  {tau:>6.2f}  {cc['pi_hat']:>7.4f}  "              f"{cc['lr_uc']:>7.3f}  {cc['p_uc']:>6.3f}  "              f"{cc['lr_ind']:>7.3f}  {cc['p_ind']:>6.3f}  "              f"{cc['lr_cc']:>7.3f}  {cc['p_cc']:>6.3f}  {esito}")        rows_cc.append({"window": window, "tau": tau, **cc})    q_cols = [f"q{int(q*100):02d}" for q in QUANTILES]    pit    = pit_test(y_te, df_pred[q_cols], QUANTILES)    print(f"\n  PIT / Berkowitz:")    print(f"    KS stat={pit['ks_stat']:.4f}  p={pit['ks_p']:.3f}"          f"  {'✓ Uniform' if pit['ks_p']>=0.05 else '✗ Non-Uniform'}")    print(f"    Berkowitz LR={pit['berk_lr']:.3f}  p={pit['berk_p']:.3f}"          f"  rho={pit['rho_hat']:.3f}  mu={pit['mu_hat']:.4f}"          f"  {'✓ iid N(0,1)' if pit['berk_p']>=0.05 else '✗ Non-iid'}")    rows_pit.append({"window": window, **pit})df_cc  = pd.DataFrame(rows_cc)df_pit = pd.DataFrame(rows_pit)df_cc.to_csv("calibration_christoffersen_2c.csv",  index=False)df_pit.to_csv("calibration_pit_berkowitz_2c.csv",  index=False)print(f"\n{'='*70}")print("RIEPILOGO — p_cc (Christoffersen CC, <0.05 = problema)")print(f"{'='*70}")piv = df_cc.pivot_table(index="window", columns="tau", values="p_cc", aggfunc="first")print(piv.round(3).to_string())print("\n── PIT / Berkowitz ──")print(df_pit[["window","ks_stat","ks_p","berk_lr","berk_p","rho_hat"]].to_string(index=False))print("\n✓ calibration_christoffersen_2c.csv")print("✓ calibration_pit_berkowitz_2c.csv")

## Stage 19 — Regime-conditional performance analysis

In [ ]:
# Metriche condizionali per stato HMM: bull / bear / high-volimport numpy as npimport pandas as pdfrom scipy import statsQUANTILES  = [0.05, 0.25, 0.50, 0.75, 0.95]WINDOWS    = ["20d", "60d", "120d"]N_STATES   = 3STATE_LABELS = {0: "State-0", 1: "State-1", 2: "State-2"}def pinball_vec(y, yhat, q):    e = y - yhat    return np.maximum(q * e, (q - 1) * e)def regime_metrics(y, pred_df, quantiles, mask):    if mask.sum() == 0:        return None    y_s  = y[mask]    q05  = pred_df["q05"].values[mask]    q25  = pred_df["q25"].values[mask]    q75  = pred_df["q75"].values[mask]    q95  = pred_df["q95"].values[mask]    pb_vals = np.mean([pinball_vec(y_s, pred_df[f"q{int(q*100):02d}"].values[mask], q)                       for q in quantiles], axis=0)    return {        "n_obs":    int(mask.sum()),        "pinball":  float(np.mean(pb_vals)),        "PICP90":   float(np.mean((y_s >= q05) & (y_s <= q95))),        "PICP50":   float(np.mean((y_s >= q25) & (y_s <= q75))),        "width90":  float(np.mean(q95 - q05)),        "viol_q05": float(np.mean(y_s < q05)),        "viol_q95": float(np.mean(y_s > q95)),        "pb_vals":  pb_vals,    }rows_regime = []for window in WINDOWS:    print(f"\n{'='*72}")    print(f"STAGE 10d | {window}  — metriche per regime HMM")    print(f"{'='*72}")    df_pred = pd.read_csv(f"final_quantile_predictions_SOFT_{window}.csv",                          parse_dates=["date"])    y_te  = df_pred["y_true"].values    dates = df_pred["date"].values    n     = len(y_te)    # Carica posteriori OOS    try:        df_post = pd.read_csv(f"hmm_proba_test_{window}_3states.csv",                              index_col=0, parse_dates=True)        df_post.columns = [f"p_state_{i}" for i in range(N_STATES)]    except FileNotFoundError:        df_post = df_pred[[f"p_state_{i}" for i in range(N_STATES)]].copy()        df_post.index = pd.to_datetime(dates)    df_post_aligned = df_post.reindex(pd.to_datetime(dates))    p_mat  = df_post_aligned[[f"p_state_{i}" for i in range(N_STATES)]].values    regime = np.argmax(p_mat, axis=1)    n_per  = [(regime == k).sum() for k in range(N_STATES)]    print(f"  Distribuzione OOS: " +          ", ".join(f"State-{k}={n_per[k]}" for k in range(N_STATES)))    print(f"\n  {'Regime':10}  {'N':>5}  {'Pinball':>9}  "          f"{'PICP90':>7}  {'PICP50':>7}  {'Width90':>8}  "          f"{'Viol_q05':>9}  {'Viol_q95':>9}")    print(f"  {'-'*72}")    per_regime_pb = {}    for k in range(N_STATES):        mask = regime == k        m    = regime_metrics(y_te, df_pred, QUANTILES, mask)        if m is None:            continue        per_regime_pb[k] = m["pb_vals"]        print(f"  {STATE_LABELS[k]:10}  {m['n_obs']:>5}  {m['pinball']:>9.4f}  "              f"{m['PICP90']:>7.3f}  {m['PICP50']:>7.3f}  {m['width90']:>8.4f}  "              f"{m['viol_q05']:>9.4f}  {m['viol_q95']:>9.4f}")        rows_regime.append({            "window": window, "state": k, "label": STATE_LABELS[k],            "n_obs": m["n_obs"], "pinball": round(m["pinball"], 5),            "PICP90": round(m["PICP90"], 4), "PICP50": round(m["PICP50"], 4),            "width90": round(m["width90"], 4),            "viol_q05": round(m["viol_q05"], 4), "viol_q95": round(m["viol_q95"], 4),        })    groups = [per_regime_pb[k] for k in sorted(per_regime_pb)              if len(per_regime_pb[k]) >= 5]    if len(groups) >= 2:        kw_stat, kw_p = stats.kruskal(*groups)        print(f"\n  Kruskal-Wallis: H={kw_stat:.3f}  p={kw_p:.4f}"              f"  {'✓ differenze significative' if kw_p < 0.05 else 'n.s.'}")    else:        kw_stat, kw_p = np.nan, np.nan        print("\n  Kruskal-Wallis: n/a")    for row in rows_regime:        if row["window"] == window and "kw_stat" not in row:            row["kw_stat"] = round(float(kw_stat), 3) if not np.isnan(kw_stat) else None            row["kw_p"]    = round(float(kw_p),    4) if not np.isnan(kw_p)    else Nonedf_reg = pd.DataFrame(rows_regime)df_reg.to_csv("regime_metrics_2d.csv", index=False)print(f"\n{'='*72}")print("RIEPILOGO — Pinball per regime")print(f"{'='*72}")print(df_reg[["window","label","n_obs","pinball","PICP90","width90",              "viol_q05","kw_p"]].to_string(index=False))print("\n✓ Salvato: regime_metrics_2d.csv")print("""NOTE:- regime = argmax(p_state_k) al tempo t  →  hard assignment- Kruskal-Wallis: testa se la pinball è uguale tra regimi- Mappa State-0/1/2 a bull/bear/high-vol guardando la media  di VIX e ritorno per regime nel set di train (STAGE 4)""")

## Stage 20 — Ablation study: LSTM without HMM featuresRe-runs the full LSTM pipeline using only the base features, to isolate the contribution of the HMM regime posteriors.

In [ ]:
# ABLATION STUDY — LSTM senza feature HMM posteriori# Riusa: WINDOWS, TARGET, BASE_FEATURES, QUANTILES, TRAIN_PROP,#        SEQ_LENGTH_GRID, UNITS_GRID, DROPOUT1_GRID, L2_GRID, LR_GRID,#        N_RANDOM_SAMPLES, N_SPLITS_TUNING, N_SPLITS, TOP_K,#        EPOCHS, EPOCHS_TUNING, BATCH_SIZE, RANDOM_STATE#        build_model(), make_sequences_all(), pinball_loss_np(),#        fold_slice_by_row_indices(), random_param_samples(),#        check_and_fix_crossing()ablation_results       = {w: [] for w in WINDOWS}ablation_ensemble_pbs  = {}for window in WINDOWS:    print("=" * 100)    print(f"ABLATION — {window} — LSTM SENZA HMM")    print("=" * 100)    target   = TARGET[window]    featcols = BASE_FEATURES[window]          # 10 feature base, NO pstate012    df = df_full[featcols + [target]].dropna().copy()    print(f"Feature: {len(featcols)} (no HMM) | Righe: {len(df)}")    splitidx = int(len(df) * TRAIN_PROP)    dftrain  = df.iloc[:splitidx].copy()    dftest   = df.iloc[splitidx:].copy()    # --- Pre-cache sequenze (come Stage 6) ---    seqcache = {}    for seq_len in SEQ_LENGTH_GRID:        Xall, yall = make_sequences_all(dftrain, featcols, target, seq_len)        seqcache[seq_len] = (Xall, yall)        print(f"  cached seq_len{seq_len} {Xall.shape} {Xall.nbytes/1024**2:.1f} MB")    # --- Stage 1: random search (come Stage 6) ---    tscv = TimeSeriesSplit(n_splits=N_SPLITS_TUNING)    rows = []    for pi, p in enumerate(random_param_samples(N_RANDOM_SAMPLES), 1):        Xall, yall = seqcache[p["seq_len"]]        nrows      = len(dftrain)        fold_losses = []        for tridx, vaidx in tscv.split(dftrain):            trseq, vaseq = fold_slice_by_row_indices(p["seq_len"], nrows, tridx, vaidx)            if len(trseq) == 0 or len(vaseq) == 0:                continue            Xtr, ytr = Xall[trseq], yall[trseq]            Xva, yva = Xall[vaseq], yall[vaseq]            sy   = StandardScaler()            ytrs = sy.fit_transform(ytr.reshape(-1, 1)).ravel()            tf.keras.backend.clear_session()            m = build_model(p["seq_len"], Xtr.shape[2], p["units"],                           p["dropout1"], p["l2"], p["lr"])            m.fit(Xtr, ytrs, epochs=EPOCHS_TUNING, batch_size=BATCH_SIZE, verbose=0)            yhatvas = m.predict(Xva, verbose=0)            yhatva  = sy.inverse_transform(yhatvas)            fold_losses.append(pinball_loss_np(yva, yhatva, QUANTILES))            del m, sy, ytrs, yhatvas, yhatva; gc.collect()        if not fold_losses:            continue        rows.append({**p,                     "nfoldsused":       len(fold_losses),                     "cv_pinball_mean":  float(np.mean(fold_losses)),                     "cv_pinball_std":   float(np.std(fold_losses))})        if pi % 10 == 0:            best_now = min(r["cv_pinball_mean"] for r in rows)            print(f"  {window} {pi}/{N_RANDOM_SAMPLES} configs best {best_now:.6g}")    cv1 = pd.DataFrame(rows).sort_values(["cv_pinball_mean", "cv_pinball_std"])    cv1.to_csv(f"ablation_cv_stage1_{window}.csv", index=False)    # --- Stage 2: top-K con N_SPLITS fold (come Stage 7) ---    tscv5     = TimeSeriesSplit(n_splits=N_SPLITS)    cv1_topk  = cv1.head(TOP_K)    needed    = sorted(cv1_topk["seq_len"].astype(int).unique().tolist())    seqcache2 = {sl: seqcache[sl] for sl in needed}    rows2 = []    for i, p in enumerate(cv1_topk.to_dict("records"), 1):        sl    = int(p["seq_len"])        Xall, yall = seqcache2[sl]        nrows = len(dftrain)        fold_losses = []        for tridx, vaidx in tscv5.split(dftrain):            trseq, vaseq = fold_slice_by_row_indices(sl, nrows, tridx, vaidx)            if len(trseq) == 0 or len(vaseq) == 0:                continue            Xtr, ytr = Xall[trseq], yall[trseq]            Xva, yva = Xall[vaseq], yall[vaseq]            sy   = StandardScaler()            ytrs = sy.fit_transform(ytr.reshape(-1, 1)).ravel()            tf.keras.backend.clear_session()            m = build_model(sl, Xtr.shape[2], p["units"],                           p["dropout1"], p["l2"], p["lr"])            m.fit(Xtr, ytrs, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)            yhatvas = m.predict(Xva, verbose=0)            yhatva  = sy.inverse_transform(yhatvas)            fold_losses.append(pinball_loss_np(yva, yhatva, QUANTILES))            del m, sy, ytrs, yhatvas, yhatva; gc.collect()        if not fold_losses:            continue        rows2.append({k: p[k] for k in ["seq_len","units","dropout1","l2","lr"]} | {            "nfoldsused":            len(fold_losses),            "stage2_cv_pinball_mean": float(np.mean(fold_losses)),            "stage2_cv_pinball_std":  float(np.std(fold_losses))})        if i % 5 == 0:            bests2 = min(r["stage2_cv_pinball_mean"] for r in rows2)            print(f"  {i}/{len(cv1_topk)} configs best {bests2:.6g}")    cv2  = pd.DataFrame(rows2).sort_values(["stage2_cv_pinball_mean","stage2_cv_pinball_std"])    cv2.to_csv(f"ablation_cv_stage2_{window}.csv", index=False)    best = cv2.iloc[0].to_dict()    print(f"BEST ablation {window}: {best}")    # --- Refit multiseed (come Stage 8.2) ---    sl = int(best["seq_len"])    Xtr, ytr = make_sequences_all(dftrain, featcols, target, sl)    Xte, yte, dte = [], [], []    for j in range(sl, len(dftest) - 1):        Xte.append(dftest[featcols].iloc[j - sl:j].values)        yte.append(dftest[target].iloc[j + 1])        dte.append(dftest.index[j + 1])    Xte = np.array(Xte, dtype=np.float32)    yte = np.array(yte, dtype=np.float32)    sy   = StandardScaler()    ytrs = sy.fit_transform(ytr.reshape(-1, 1)).ravel()    all_yhats, seed_pinballs = [], []    for seed in SEEDS:        np.random.seed(seed); tf.random.set_seed(seed)        tf.keras.backend.clear_session()        m = build_model(sl, Xtr.shape[2], best["units"],                       best["dropout1"], best["l2"], best["lr"])        m.fit(Xtr, ytrs, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)        yhats = m.predict(Xte, verbose=0)        yhat  = sy.inverse_transform(yhats)        yhat  = check_and_fix_crossing(yhat, QUANTILES, f"ablation_{window}")        pb    = pinball_loss_np(yte, yhat, QUANTILES)        seed_pinballs.append(pb)        all_yhats.append(yhat.copy())        print(f"  seed {seed:>3} pinball {pb:.6f}")        del m, yhats; gc.collect()    ensemble_yhat = np.mean(all_yhats, axis=0)    ensemble_pb   = pinball_loss_np(yte, ensemble_yhat, QUANTILES)    ablation_ensemble_pbs[window] = ensemble_pb    out = pd.DataFrame(ensemble_yhat,                       columns=[f"q{int(q*1000):04d}" for q in QUANTILES])    out.insert(0, "date", dte)    out["ytrue"] = yte    out.to_csv(f"ablation_predictions_{window}.csv", index=False)    arr = np.array(seed_pinballs)    ablation_results[window] = arr    print(f"\n{window} ablation | mean {arr.mean():.6f} \u00b1 {arr.std():.6f} | ensemble {ensemble_pb:.6f}")    del seqcache; gc.collect()# Riepilogopd.DataFrame([{    "window":                    w,    "ablation_pinball_mean":     ablation_results[w].mean(),    "ablation_pinball_std":      ablation_results[w].std(),    "ablation_ensemble_pinball": ablation_ensemble_pbs[w]} for w in WINDOWS]).to_csv("ablation_summary.csv", index=False)print("\n\u2705 Salvato ablation_summary.csv")

## Stage 21 — Ablation comparison (Diebold-Mariano)

In [ ]:
# STAGE 11.2 — CONFRONTO HMM-LSTM vs ABLATION (robusto)# Usa i file realmente salvati nel notebook:# - final_quantile_predictions_SOFT_{window}.csv# - ablation_predictions_{window}.csv# Salva:# - ablation_comparison_DM.csvimport osimport numpy as npimport pandas as pdfrom scipy import stats# Helper robusti per colonne e nomi filedef get_quantile_colnames_from_df(df, quantiles):    """    Ritorna il mapping {q: colname} cercando varianti possibili:    q05 / q25 / q50 / q75 / q95    q0050 / q0250 / ...    """    candidates = {}    for q in quantiles:        c1 = f"q{int(q*100):02d}"      # es. q05, q25, q50...        c2 = f"q{int(q*1000):04d}"     # es. q0050, q0250...        if c1 in df.columns:            candidates[q] = c1        elif c2 in df.columns:            candidates[q] = c2        else:            raise KeyError(f"Colonna quantile non trovata per q={q}. Cercate: {c1}, {c2}")    return candidatesdef get_y_colname(df):    """    Cerca la colonna del target reale.    """    for c in ["y_true", "ytrue", "target", "actual"]:        if c in df.columns:            return c    raise KeyError("Nessuna colonna target trovata. Attese: y_true / ytrue / target / actual")def get_date_colname(df):    """    Cerca la colonna data.    """    for c in ["date", "Date", "datetime"]:        if c in df.columns:            return c    raise KeyError("Nessuna colonna data trovata. Attese: date / Date / datetime")def pinball_per_obs(y, yhat, quantiles):    """    Pinball loss media per osservazione, shape output = (n,)    """    y = np.asarray(y).reshape(-1, 1)    losses = []    for j, q in enumerate(quantiles):        e = y[:, 0] - yhat[:, j]        losses.append(np.maximum(q * e, (q - 1) * e))    return np.mean(np.column_stack(losses), axis=1)def dm_test_hln(loss_model1, loss_model2):    """    Diebold-Mariano con correzione Harvey-Leybourne-Newbold.    H0: stesso expected loss    loss_model1 - loss_model2 < 0 => model1 migliore    """    d = np.asarray(loss_model1) - np.asarray(loss_model2)    d = d[np.isfinite(d)]    n = len(d)    if n < 10:        return {            "DM_stat": np.nan,            "HLN_stat": np.nan,            "p_value": np.nan,            "reject_H0": False,            "better_model": "n.a."        }    d_mean = d.mean()    gamma0 = np.var(d, ddof=1)    if n > 1:        gamma1 = np.cov(d[:-1], d[1:], ddof=1)[0, 1]    else:        gamma1 = 0.0    var_d = (gamma0 + 2 * gamma1) / n    var_d = max(var_d, 1e-12)    dm_stat = d_mean / np.sqrt(var_d)    # HLN correction, horizon h=1    h = 1    hln_factor = np.sqrt((n + 1 - 2*h + h*(h-1)/n) / n)    hln_stat = dm_stat * hln_factor    p_value = float(2 * (1 - stats.t.cdf(abs(hln_stat), df=n - 1)))    if p_value < 0.05:        better = "HMM-LSTM" if dm_stat < 0 else "LSTM-only"    else:        better = "n.s."    return {        "DM_stat": float(dm_stat),        "HLN_stat": float(hln_stat),        "p_value": float(p_value),        "reject_H0": bool(p_value < 0.05),        "better_model": better    }# Loop principalerows = []print("=" * 95)print("STAGE 11.2 — CONFRONTO HMM-LSTM vs LSTM-only ABLATION")print("=" * 95)print(f"{'window':<8} {'pin_HMM':>10} {'pin_ABL':>10} {'delta':>10} {'HLN':>10} {'p-value':>10} {'better':>12}")print("-" * 95)for window in WINDOWS:    file_hmm = f"final_quantile_predictions_SOFT_{window}.csv"    file_abl = f"ablation_predictions_{window}.csv"    if not os.path.exists(file_hmm):        print(f"{window:<8} SKIP  file mancante: {file_hmm}")        continue    if not os.path.exists(file_abl):        print(f"{window:<8} SKIP  file mancante: {file_abl}")        continue    df_hmm = pd.read_csv(file_hmm)    df_abl = pd.read_csv(file_abl)    date_hmm = get_date_colname(df_hmm)    date_abl = get_date_colname(df_abl)    y_hmm_col = get_y_colname(df_hmm)    y_abl_col = get_y_colname(df_abl)    qmap_hmm = get_quantile_colnames_from_df(df_hmm, QUANTILES)    qmap_abl = get_quantile_colnames_from_df(df_abl, QUANTILES)    df_hmm[date_hmm] = pd.to_datetime(df_hmm[date_hmm])    df_abl[date_abl] = pd.to_datetime(df_abl[date_abl])    df_hmm = df_hmm[[date_hmm, y_hmm_col] + [qmap_hmm[q] for q in QUANTILES]].copy()    df_abl = df_abl[[date_abl, y_abl_col] + [qmap_abl[q] for q in QUANTILES]].copy()    df_hmm.columns = ["date", "y_true"] + [f"q{int(q*100):02d}" for q in QUANTILES]    df_abl.columns = ["date", "y_true"] + [f"q{int(q*100):02d}" for q in QUANTILES]    df_cmp = (        df_hmm.merge(df_abl, on="date", how="inner", suffixes=("_hmm", "_abl"))              .sort_values("date")              .reset_index(drop=True)    )    if df_cmp.empty:        print(f"{window:<8} SKIP  nessuna data comune")        continue    y = df_cmp["y_true_hmm"].values    y_abl = df_cmp["y_true_abl"].values    if not np.allclose(y, y_abl, equal_nan=True):        print(f"{window:<8} ATTENZIONE: y_true non coincide perfettamente, uso y_true_hmm")    qcols = [f"q{int(q*100):02d}" for q in QUANTILES]    pred_hmm = df_cmp[[f"{c}_hmm" for c in qcols]].copy()    pred_abl = df_cmp[[f"{c}_abl" for c in qcols]].copy()    pred_hmm.columns = qcols    pred_abl.columns = qcols    yhat_hmm = pred_hmm.values    yhat_abl = pred_abl.values    pb_hmm = pinball_loss_np(y, yhat_hmm, QUANTILES)    pb_abl = pinball_loss_np(y, yhat_abl, QUANTILES)    delta = pb_hmm - pb_abl    loss_hmm = pinball_per_obs(y, yhat_hmm, QUANTILES)    loss_abl = pinball_per_obs(y, yhat_abl, QUANTILES)    dm = dm_test_hln(loss_hmm, loss_abl)    q05_hmm = pred_hmm["q05"].values    q95_hmm = pred_hmm["q95"].values    q05_abl = pred_abl["q05"].values    q95_abl = pred_abl["q95"].values    picp90_hmm = float(np.mean((y >= q05_hmm) & (y <= q95_hmm)))    picp90_abl = float(np.mean((y >= q05_abl) & (y <= q95_abl)))    width90_hmm = float(np.mean(q95_hmm - q05_hmm))    width90_abl = float(np.mean(q95_abl - q05_abl))    print(f"{window:<8} {pb_hmm:>10.6f} {pb_abl:>10.6f} {delta:>+10.6f} "          f"{dm['HLN_stat']:>10.4f} {dm['p_value']:>10.4f} {dm['better_model']:>12}")    rows.append({        "window": window,        "n_common_dates": len(df_cmp),        "pinball_HMM": round(pb_hmm, 6),        "pinball_ablation": round(pb_abl, 6),        "delta_pinball_HMM_minus_ablation": round(delta, 6),        "PICP90_HMM": round(picp90_hmm, 4),        "PICP90_ablation": round(picp90_abl, 4),        "width90_HMM": round(width90_hmm, 6),        "width90_ablation": round(width90_abl, 6),        "DM_stat": round(dm["DM_stat"], 4) if np.isfinite(dm["DM_stat"]) else np.nan,        "HLN_stat": round(dm["HLN_stat"], 4) if np.isfinite(dm["HLN_stat"]) else np.nan,        "p_value_DM": round(dm["p_value"], 4) if np.isfinite(dm["p_value"]) else np.nan,        "reject_H0": dm["reject_H0"],        "better_model": dm["better_model"]    })df_out = pd.DataFrame(rows)df_out.to_csv("ablation_comparison_DM.csv", index=False)print("-" * 95)print("✓ Salvato: ablation_comparison_DM.csv")display(df_out)

## Stage 22 — Economic evaluation

In [ ]:
# STAGE 12 - VALUTAZIONE ECONOMICA CORRETTA# - usa solo file realmente generati nel notebook# - HQ ricostruito dal train# - GARCH-t letto da baseline_garcht_preds_{window}.csv# - metriche economiche PER ORIZZONTE (non annualizzate)import osimport gcimport numpy as npimport pandas as pdfrom scipy import statsWINDOWS = ["20d", "60d", "120d"]QUANTILES = [0.05, 0.25, 0.50, 0.75, 0.95]HORIZON_DAYS = {"20d": 20, "60d": 60, "120d": 120}# Helperdef qcol(q):    return f"q{int(q*100):02d}"def pinball_pointwise(y, yhat, q):    e = y - yhat    return np.maximum(q * e, (q - 1) * e)def pinball_mean(y, pred_df, quantiles=QUANTILES):    vals = []    for q in quantiles:        vals.append(pinball_pointwise(y, pred_df[qcol(q)].values, q).mean())    return float(np.mean(vals))def winkler_score(y, q_lo, q_hi, alpha):    width = q_hi - q_lo    below = y < q_lo    above = y > q_hi    penalty = np.where(        below, 2 / alpha * (q_lo - y),        np.where(above, 2 / alpha * (y - q_hi), 0.0)    )    return float(np.mean(width + penalty))def kupiec_test_left(y, q_hat, tau=0.05):    n = len(y)    hits = np.sum(y < q_hat)    tau_hat = hits / n    tau_hat_clip = np.clip(tau_hat, 1e-10, 1 - 1e-10)    tau_clip = np.clip(tau, 1e-10, 1 - 1e-10)    lr = 2 * (        hits * np.log(tau_hat_clip / tau_clip) +        (n - hits) * np.log((1 - tau_hat_clip) / (1 - tau_clip))    )    p_val = 1 - stats.chi2.cdf(lr, df=1)    return float(lr), float(p_val), float(tau_hat)def christoffersen_test(y, q_hat, tau=0.05):    hits = (y < q_hat).astype(int)    n = len(hits)    n1 = hits.sum()    n0 = n - n1    pihat = n1 / n    if pihat == 0 or pihat == 1:        lr_uc, p_uc = np.nan, np.nan    else:        lr_uc = 2 * (            n1 * np.log(pihat / tau) +            n0 * np.log((1 - pihat) / (1 - tau))        )        p_uc = 1 - stats.chi2.cdf(lr_uc, df=1)    n00 = ((hits[:-1] == 0) & (hits[1:] == 0)).sum()    n01 = ((hits[:-1] == 0) & (hits[1:] == 1)).sum()    n10 = ((hits[:-1] == 1) & (hits[1:] == 0)).sum()    n11 = ((hits[:-1] == 1) & (hits[1:] == 1)).sum()    pi01 = n01 / (n00 + n01) if (n00 + n01) > 0 else 0.0    pi11 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0.0    pi2 = (n01 + n11) / max(n00 + n01 + n10 + n11, 1)    def safe_log(x):        return np.log(x) if x > 0 else 0.0    lr_ind = 2 * (        n00 * safe_log(1 - pi01) + n01 * safe_log(pi01) +        n10 * safe_log(1 - pi11) + n11 * safe_log(pi11) -        (n00 + n10) * safe_log(1 - pi2) -        (n01 + n11) * safe_log(pi2)    )    p_ind = 1 - stats.chi2.cdf(lr_ind, df=1)    lr_cc = lr_uc + lr_ind if not np.isnan(lr_uc) else np.nan    p_cc = 1 - stats.chi2.cdf(lr_cc, df=2) if not np.isnan(lr_cc) else np.nan    return {        "lr_uc": float(lr_uc) if not np.isnan(lr_uc) else np.nan,        "p_uc": float(p_uc) if not np.isnan(lr_uc) else np.nan,        "lr_ind": float(lr_ind),        "p_ind": float(p_ind),        "lr_cc": float(lr_cc) if not np.isnan(lr_uc) else np.nan,        "p_cc": float(p_cc) if not np.isnan(lr_uc) else np.nan,    }def dm_test_quantiles(y, pred_a, pred_b, quantiles=QUANTILES):    d = np.zeros(len(y))    for q in quantiles:        col = qcol(q)        la = pinball_pointwise(y, pred_a[col].values, q)        lb = pinball_pointwise(y, pred_b[col].values, q)        d += (lb - la)    d /= len(quantiles)    n = len(d)    d_bar = np.mean(d)    h = max(1, int(n ** (1 / 3)))    gamma0 = np.var(d, ddof=1)    nw_var = gamma0    for k in range(1, h + 1):        gamma_k = np.cov(d[k:], d[:-k])[0, 1]        nw_var += 2 * (1 - k / (h + 1)) * gamma_k    nw_var = max(nw_var, 1e-10)    dm_stat = d_bar / np.sqrt(nw_var / n)    p_val = 1 - stats.norm.cdf(dm_stat)   # one-sided: A better than B    return float(dm_stat), float(p_val)def build_hq_preds_from_train(y_train, n_test, quantiles=QUANTILES):    data = {}    for q in quantiles:        data[qcol(q)] = np.full(n_test, np.quantile(y_train, q))    return pd.DataFrame(data)def load_garcht_preds(window):    fname = f"baseline_garcht_preds_{window}.csv"    if not os.path.exists(fname):        raise FileNotFoundError(f"File non trovato: {fname}")    df = pd.read_csv(fname)    needed = [qcol(q) for q in QUANTILES]    missing = [c for c in needed if c not in df.columns]    if missing:        raise ValueError(f"{fname} manca le colonne: {missing}")    return df[needed].copy()def strategy_stats_per_horizon(returns_h, weights):    """    returns_h: rendimenti target su orizzonte h (20d/60d/120d)    weights: esposizione in [0.5, 1.0]    NIENTE annualizzazione: metriche coerenti col target del notebook.    """    strat_ret = weights * returns_h    mean_ret_h = np.mean(strat_ret)    vol_h = np.std(strat_ret, ddof=1)    sharpe_h = mean_ret_h / vol_h if vol_h > 1e-12 else np.nan    cum = np.cumprod(1 + strat_ret)    peak = np.maximum.accumulate(cum)    dd = cum / peak - 1.0    max_dd = np.min(dd)    turnover = np.mean(np.abs(np.diff(weights))) if len(weights) > 1 else 0.0    return {        "mean_ret_h": float(mean_ret_h),        "vol_h": float(vol_h),        "sharpe_h": float(sharpe_h) if not np.isnan(sharpe_h) else np.nan,        "max_dd": float(max_dd),        "turnover": float(turnover),        "avg_weight": float(np.mean(weights)),    }# Loop principaleall_metrics = []all_strategy = []for window in WINDOWS:    print("\n" + "=" * 90)    print(f"STAGE 12 | {window}")    print("=" * 90)    target = TARGET[window]    base_cols = BASE_FEATURES[window]    soft_cols = [f"p_state_{i}" for i in range(N_STATES)]    # Ricostruzione identica al protocollo già usato in Stage 8 / 8.2    posteriors = load_hmm_posteriors(window)    df = df_full[base_cols + [target]].dropna().copy()    df = add_soft_states(df, posteriors)    split_idx = int(len(df) * TRAIN_PROP)    df_train = df.iloc[:split_idx].copy()    df_test = df.iloc[split_idx:].copy()    best = pd.read_csv(f"walkforward_quantile_cv_stage2_SOFT_{window}.csv").iloc[0].to_dict()    sl = int(best["seq_len"])    # Allineamento OOS identico a Stage 8/8.2    y_test = []    d_test = []    for j in range(sl, len(df_test) - 1):        y_test.append(df_test[target].iloc[j + 1])        d_test.append(df_test.index[j + 1])    y_test = np.array(y_test, dtype=np.float32)    # HMM-LSTM    lstm = pd.read_csv(f"final_quantile_predictions_SOFT_{window}.csv")    lstm = lstm[[qcol(q) for q in QUANTILES] + ["y_true"]].copy()    if len(lstm) != len(y_test):        raise ValueError(            f"[{window}] mismatch: file LSTM con {len(lstm)} righe vs "            f"target ricostruito con {len(y_test)} righe."        )    # HQ da train    hq = build_hq_preds_from_train(df_train[target].values, len(y_test), QUANTILES)    # GARCH-t da file effettivamente salvato    garcht = load_garcht_preds(window)    if len(garcht) != len(y_test):        raise ValueError(            f"[{window}] mismatch: GARCH-t con {len(garcht)} righe vs target {len(y_test)}."        )    print(f"GARCH-t caricato da baseline_garcht_preds_{window}.csv")    model_map = {        "HMM-LSTM": lstm[[qcol(q) for q in QUANTILES]].copy(),        "HQ": hq.copy(),        "GARCH-t": garcht.copy(),    }    # Metriche rischio / calibrazione    for model_name, pred in model_map.items():        q05 = pred["q05"].values        q25 = pred["q25"].values        q75 = pred["q75"].values        q95 = pred["q95"].values        pb = pinball_mean(y_test, pred)        picp90 = np.mean((y_test >= q05) & (y_test <= q95))        picp50 = np.mean((y_test >= q25) & (y_test <= q75))        width90 = np.mean(q95 - q05)        width50 = np.mean(q75 - q25)        wink90 = winkler_score(y_test, q05, q95, alpha=0.10)        wink50 = winkler_score(y_test, q25, q75, alpha=0.50)        kup_lr, kup_p, breach = kupiec_test_left(y_test, q05, tau=0.05)        cc = christoffersen_test(y_test, q05, tau=0.05)        all_metrics.append({            "window": window,            "model": model_name,            "pinball": float(pb),            "PICP90": float(picp90),            "PICP50": float(picp50),            "width90": float(width90),            "width50": float(width50),            "winkler90": float(wink90),            "winkler50": float(wink50),            "VaR_breach_rate": float(breach),            "kupiec_LR": float(kup_lr),            "kupiec_p": float(kup_p),            "cc_lr_uc": cc["lr_uc"],            "cc_p_uc": cc["p_uc"],            "cc_lr_ind": cc["lr_ind"],            "cc_p_ind": cc["p_ind"],            "cc_lr_cc": cc["lr_cc"],            "cc_p_cc": cc["p_cc"],        })    # DM test: HMM-LSTM vs baseline    ref = model_map["HMM-LSTM"]    for other_name in ["HQ", "GARCH-t"]:        dm_stat, dm_p = dm_test_quantiles(y_test, ref, model_map[other_name], QUANTILES)        all_metrics.append({            "window": window,            "model": f"DM: HMM-LSTM vs {other_name}",            "pinball": np.nan,            "PICP90": np.nan,            "PICP50": np.nan,            "width90": np.nan,            "width50": np.nan,            "winkler90": np.nan,            "winkler50": np.nan,            "VaR_breach_rate": np.nan,            "kupiec_LR": dm_stat,            "kupiec_p": dm_p,            "cc_lr_uc": np.nan,            "cc_p_uc": np.nan,            "cc_lr_ind": np.nan,            "cc_p_ind": np.nan,            "cc_lr_cc": np.nan,            "cc_p_cc": np.nan,        })    # Strategia economica semplice basata su VaR q05    # Regola:    # se q05 è sopra la mediana della sua serie -> peso 1    # altrimenti -> peso 0.5    # Metriche per orizzonte, NON annualizzate    for model_name, pred in model_map.items():        q05 = pred["q05"].values        threshold = np.median(q05)        weights = np.where(q05 >= threshold, 1.0, 0.5).astype(float)        st = strategy_stats_per_horizon(y_test, weights)        all_strategy.append({            "window": window,            "horizon_days": HORIZON_DAYS[window],            "model": model_name,            "threshold_q05_median": float(threshold),            "mean_ret_h": st["mean_ret_h"],            "vol_h": st["vol_h"],            "sharpe_h": st["sharpe_h"],            "max_dd": st["max_dd"],            "turnover": st["turnover"],            "avg_weight": st["avg_weight"],        })    gc.collect()# Salvataggiometrics_df = pd.DataFrame(all_metrics)strategy_df = pd.DataFrame(all_strategy)metrics_df.to_csv("stage12_economic_metrics.csv", index=False)strategy_df.to_csv("stage12_var_strategy.csv", index=False)print("\n" + "=" * 90)print("SALVATAGGI STAGE 12")print("=" * 90)print("→ Salvato: stage12_economic_metrics.csv")print("→ Salvato: stage12_var_strategy.csv")metrics = pd.read_csv("stage12_economic_metrics.csv")strategy = pd.read_csv("stage12_var_strategy.csv")display(metrics)display(strategy)print("\nPinball per modello/orizzonte")display(    metrics[metrics["model"].isin(["HMM-LSTM", "HQ", "GARCH-t"])]    .pivot(index="window", columns="model", values="pinball"))print("\nSharpe per orizzonte")display(    strategy.pivot(index="window", columns="model", values="sharpe_h"))